In [ ]:
# =============================================================================
# NOTEBOOK: transform_partner_to_gold_v1.py
# MỤC ĐÍCH : Đọc Bronze partner data → transform → ghi 3 bảng Gold:
#             poi_master · ai_poi · poi_detail
#
# ĐIỂM KHÁC SO VỚI SQL CŨ:
#   1. Đọc bronze 1 lần duy nhất (SQL đọc 3 lần)
#   2. parse_amentities() 1 lần / record (SQL gọi get_json_object 162 lần)
#   3. Watermark: chỉ xử lý services thay đổi kể từ lần chạy trước
#   4. Test data factory: tạo fixture để test từng hàm riêng
#   5. Mỗi transform tách thành hàm riêng → dễ test, dễ sửa từng sector
#
# CẤU TRÚC NOTEBOOK (10 cell):
#   CELL 1  — Import + config
#   CELL 2  — Hàm tiện ích + parse JSON
#   CELL 3  — Sector config (amenities, policy, hours per sector)
#   CELL 4  — Watermark: đọc + tính cutoff
#   CELL 5  — Đọc bronze (1 lần), join, enrich
#   CELL 6  — Transform functions (poi_master, ai_poi, poi_detail rows)
#   CELL 7  — Build DataFrames
#   CELL 8  — MERGE 3 bảng Gold
#   CELL 9  — Update watermark
#   CELL 10 — Test data factory + unit tests
# =============================================================================


# %%
# =============================================================================
# CELL 1 — IMPORT VÀ CONFIG
# =============================================================================

import json
import re
import traceback
import uuid
from datetime import datetime
from hashlib import md5
from typing import Any

from delta.tables import DeltaTable
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    BooleanType, DecimalType, DoubleType, IntegerType,
    LongType, StringType, StructField, StructType, TimestampType,
)
from pyspark.storagelevel import StorageLevel

# --------------------------------------------------------------------------
# Tables
# --------------------------------------------------------------------------
BRZ_SERVICES   = "lh_vv_bronze.partner.business_services"
BRZ_PARTNERS   = "lh_vv_bronze.partner.partners"
BRZ_LOCATIONS  = "lh_vv_bronze.partner.business_locations"
BRZ_I18NS      = "lh_vv_bronze.partner.business_service_i18ns"
GLD_DEST_MAP   = "lh_vv_gold.dbo.destination_ward_mapping"
GLD_AI_PRODUCT = "lh_vv_gold.dbo.ai_product"
GLD_POI_ADDR   = "lh_vv_gold.dbo.poi_address_enrichment_staging"

GLD_POI_MASTER = "lh_vv_gold.dbo.poi_master"
GLD_AI_POI     = "lh_vv_gold.dbo.ai_poi"
GLD_POI_DETAIL = "lh_vv_gold.dbo.poi_detail"

# Watermark: lưu max(last_updated_at) đã xử lý thành công
WM_TABLE   = "lh_vv_gold.dbo.partner_gold_watermark"
WM_FLOW_ID = "transform_partner_to_gold_v1"

RUN_ID     = str(uuid.uuid4())
INGESTION  = "partner_portal"
SENTINEL   = "__debezium_unavailable_value"

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")

print(f"RUN_ID: {RUN_ID}")


# %%
# =============================================================================
# CELL 2 — HÀM TIỆN ÍCH + PARSE JSON
# =============================================================================
# Giải thích:
#   parse_amentities() gọi 1 lần / record, trả về dict Python.
#   Các hàm extract_* sau đó làm dict lookup O(1) thay vì parse JSON lại.
#   SQL cũ gọi get_json_object 162 lần trên cùng 1 cột → 162× overhead.
# =============================================================================

TOAST = SENTINEL  # alias ngắn hơn

def normalize_city_name(col_or_name: Any):
    """
    Chuyển thường, trim và xóa các tiền tố hành chính phổ biến
    Chấp nhận tham số là tên cột (str) hoặc đối tượng Spark Column.
    """
    c = col_or_name if isinstance(col_or_name, F.Column) else F.col(col_or_name)
    cleaned = F.lower(F.trim(F.coalesce(c, F.lit(""))))
    # Bỏ tiền tố cấp tỉnh/thành
    cleaned = F.regexp_replace(cleaned, r"^(thành phố|tỉnh|tp\.?)\s+", "")
    # Bỏ tiền tố cấp phường/xã/thị trấn
    cleaned = F.regexp_replace(cleaned, r"^(phường|xã|thị trấn|tt\.?)\s+", "")
    return F.trim(cleaned)

def clean(v: Any) -> Any:
    """Trả về None nếu giá trị là sentinel hoặc chuỗi rỗng."""
    if v is None:
        return None
    s = str(v).strip()
    if s == "" or s == TOAST:
        return None
    return s


def parse_json(raw: str | None) -> dict:
    """
    Parse JSON string thành dict.
    Trả về {} nếu None, sentinel, JSON không hợp lệ, hoặc JSON là array/scalar.
    Tránh lỗi AttributeError khi gọi .get() trên list hoặc None.
    """
    s = clean(raw)
    if s is None:
        return {}
    try:
        result = json.loads(s)
        # Chỉ trả về dict — nếu parse ra list/int/str thì trả về {}
        return result if isinstance(result, dict) else {}
    except (json.JSONDecodeError, TypeError):
        return {}


def make_poi_id(source_id: str, lang: str = "vi") -> str:
    """
    Tạo poi_id dạng UUID từ (ingestion_source, source_id, lang).
    Deterministic: cùng input luôn ra cùng ID.
    Giống SQL: regexp_replace(md5(concat(...)), '^(.{8})...')
    """
    raw = md5(f"{INGESTION}{source_id}{lang}".encode()).hexdigest()
    return f"{raw[:8]}-{raw[8:12]}-{raw[12:16]}-{raw[16:20]}-{raw[20:32]}"


def make_slug(name: str, poi_id: str) -> str:
    """Slug = tên latinh hóa + 8 ký tự cuối poi_id."""
    latin = re.sub(r"[^a-z0-9]+", "-",
                   re.sub(r"[^\x00-\x7F]", "", (name or "").lower()))
    latin = latin.strip("-") or ""
    return f"{latin}-{poi_id[-8:]}"


def epoch_ms_to_ts(epoch_ms: int | None) -> datetime | None:
    """
    Epoch seconds hoặc milliseconds → datetime.
    Tự detect: nếu giá trị > 9_999_999_999 (năm 2286) thì coi là milliseconds.
    Trả về None nếu None hoặc 0.
    """
    if not epoch_ms:
        return None
    try:
        v = int(epoch_ms)
        # > 10 chữ số = milliseconds → chia 1000
        if v > 9_999_999_999:
            v = v // 1000
        return datetime.utcfromtimestamp(v)
    except (ValueError, OSError):
        return None


def nullif_empty(v: Any) -> Any:
    """Giống NULLIF(v, '') trong SQL."""
    if v is None:
        return None
    s = str(v).strip()
    return None if s == "" else s


def json_list_lower(raw: str | None) -> str:
    """Trả về JSON array string dạng lowercase — dùng để RLIKE."""
    v = clean(raw)
    return (v or "").lower()


def contains_any(text: str, keywords: list[str]) -> bool:
    """True nếu text chứa ít nhất 1 keyword (case-insensitive)."""
    t = text.lower()
    return any(k in t for k in keywords)


print("✓ Cell 2: utility functions ready")


# %%
# =============================================================================
# CELL 3 — SECTOR CONFIG
# =============================================================================
# Giải thích:
#   Thay vì viết CASE WHEN business_sector = 'f_n_b' THEN ... lặp 20 lần,
#   ta khai báo config 1 lần ở đây.
#   Khi thêm sector mới (golf_resort, marina...) chỉ cần thêm 1 dict.
#
#   Mỗi sector config gồm:
#     category_path  : path trong amentities dict để lấy business_category
#     specialty_path : path để lấy specialty_tags
#     amenity_key    : key trong facilities để lấy general_amenities
#     oh_path        : path đến operating_hours
#     cancel_path    : path đến cancellation_policy
# =============================================================================

SECTOR_CONFIG = {
    "accommodation": {
        "category_path":    ["facilities", "main_category"],
        "specialty_path":   None,
        "amenity_key":      ["facilities", "general_amenities"],
        "oh_path":          ["policies", "operating_hours"],
        "cancel_path":      ["policies", "cancellation_policy"],
        "pet_path":         ["policies", "pet_policy"],
        "venue_reg_path":   None,
    },
    "f_n_b": {
        "category_path":    ["facilities", "main_cuisine"],
        "specialty_path":   ["facilities", "specialty_tags"],     # [F6] đã sửa
        "amenity_key":      ["facilities", "space_and_services"],
        "oh_path":          ["policies", "operating_hours"],
        "cancel_path":      ["policies", "deposit_and_cancellation"],
        "pet_path":         ["policies", "house_rules"],
        "venue_reg_path":   None,
    },
    "spa_n_wellness": {
        "category_path":    ["center_profile", "business_category"],
        "specialty_path":   None,
        "amenity_key":      ["center_profile", "facilities_and_ambience"],
        "oh_path":          ["policies", "operating_hours"],
        "cancel_path":      ["policies", "cancellation_policy"],
        "pet_path":         None,
        "venue_reg_path":   ["policies", "health_restrictions"],
    },
    "shopping_n_retail": {
        "category_path":    ["store_profile", "main_category"],
        "specialty_path":   None,
        "amenity_key":      ["facilities", "store_services"],
        "oh_path":          ["policies", "operating_hours"],
        "cancel_path":      ["policies", "cancellation_policy"],
        "pet_path":         None,
        "venue_reg_path":   None,
    },
    "attraction": {
        "category_path":    ["facilities", "classification", "main_category"],
        "specialty_path":   ["facilities", "classification", "specialty_tags"],  # [F6]
        "amenity_key":      ["facilities", "facilities_and_services"],
        "oh_path":          ["policies", "operating_hours"],
        "cancel_path":      ["policies", "refund_and_change_policy"],
        "pet_path":         None,
        "venue_reg_path":   None,  # attraction dùng gate_and_entry_rules riêng
    },
    "transport_on_demand": {
        "category_path":    ["operator_profile", "business_model"],
        "specialty_path":   None,
        "amenity_key":      None,
        "oh_path":          ["policies", "operating_hours"],
        "cancel_path":      ["policies", "cancellation_policy"],
        "pet_path":         None,
        "venue_reg_path":   None,
    },
    "transport_route_base": {
        "category_path":    ["fleet_profile", "vehicle_types"],
        "specialty_path":   None,
        "amenity_key":      None,
        "oh_path":          ["policies", "operating_hours"],
        "cancel_path":      ["policies", "cancellation_policy"],
        "pet_path":         None,
        "venue_reg_path":   None,
    },
    "event": {
        "category_path":    ["organizer_profile", "core_competency"],
        "specialty_path":   None,
        "amenity_key":      ["facilities", "event_services"],
        "oh_path":          ["policies", "operating_hours"],
        "cancel_path":      ["policies", "refund_and_postponement"],
        "pet_path":         None,
        "venue_reg_path":   ["policies", "venue_regulations"],
    },
    "vehicle_rental": {
        "category_path":    ["logistics", "business_type"],
        "specialty_path":   None,
        "amenity_key":      None,
        "oh_path":          ["policies", "operating_hours"],
        "cancel_path":      ["policies", "cancellation_policy"],
        "pet_path":         None,
        "venue_reg_path":   None,
    },
    "other": {
        "category_path":    None,
        "specialty_path":   None,
        "amenity_key":      None,
        "oh_path":          ["policies", "operating_hours"],
        "cancel_path":      ["policies", "cancellation_policy"],
        "pet_path":         None,
        "venue_reg_path":   None,
    },
    "tour_activities": {
        "category_path":    None,
        "specialty_path":   None,
        "amenity_key":      None,
        "oh_path":          ["policies", "operating_hours"],
        "cancel_path":      ["policies", "cancellation_policy"],
        "pet_path":         None,
        "venue_reg_path":   None,
    },
}

# Default config cho sector chưa khai báo
DEFAULT_SECTOR_CONFIG = {
    "category_path":    None,
    "specialty_path":   None,
    "amenity_key":      None,
    "oh_path":          ["policies", "operating_hours"],
    "cancel_path":      ["policies", "cancellation_policy"],
    "pet_path":         None,
    "venue_reg_path":   None,
}


def get_sector_cfg(sector: str) -> dict:
    """Lấy config của sector, fallback về DEFAULT nếu chưa khai báo."""
    return SECTOR_CONFIG.get(sector, DEFAULT_SECTOR_CONFIG)


def deep_get(d: dict, *keys) -> Any:
    """
    Truy cập dict lồng nhau theo path.
    deep_get(d, 'policies', 'operating_hours') = d['policies']['operating_hours']
    Trả về None nếu bất kỳ key nào không tồn tại.
    """
    cur = d
    for k in keys:
        if not isinstance(cur, dict):
            return None
        cur = cur.get(k)
    return cur


print("✓ Cell 3: sector configs ready")


# %%
# =============================================================================
# CELL 4 — WATERMARK: ĐỌC + TÍNH CUTOFF
# =============================================================================
# Giải thích:
#   Watermark lưu max(last_updated_at) của lần chạy trước.
#   Lần này chỉ đọc services có last_updated_at > watermark.
#   Nếu bảng watermark chưa có → chạy full (cutoff = 0).
#
#   Khác với Bronze notebook (dùng EventProcessedUtcTime của raw table),
#   Gold dùng last_updated_at của business_services — đây là timestamp
#   nghiệp vụ thật sự của record.
# =============================================================================


# --------------------------------------------------------------------------
# Helper: reset watermark để force full load lại từ đầu
# Bỏ comment 2 dòng dưới rồi chạy cell này nếu muốn chạy lại toàn bộ
# # --------------------------------------------------------------------------
# spark.sql(f"DELETE FROM {WM_TABLE} WHERE flow_id = '{WM_FLOW_ID}'")
# print("✓ Watermark reset xong — chạy lại toàn bộ notebook từ Cell 1")


# Tạo bảng watermark nếu chưa có
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {WM_TABLE} (
    flow_id         STRING,
    last_updated_at BIGINT,
    last_run_id     STRING,
    last_run_at     TIMESTAMP,
    status          STRING
) USING DELTA
""")

# Đọc watermark
wm_row = (
    spark.table(WM_TABLE)
    .filter(F.col("flow_id") == WM_FLOW_ID)
    .filter(F.col("status") == "SUCCESS")
    .orderBy(F.col("last_updated_at").desc())
    .limit(1)
    .first()
)

# cutoff = epoch seconds của lần chạy trước
# Nếu chưa có watermark → cutoff = 0 → đọc toàn bộ (backfill)
cutoff_s = int(wm_row["last_updated_at"]) if wm_row else 0

# effective_cutoff = cutoff - 1 giây để include records có last_updated_at == cutoff
# Tránh bỏ sót records bằng đúng watermark value
# Bỏ comment dòng dưới nếu muốn force full load:
# cutoff_s = 0
effective_cutoff = max(0, cutoff_s - 1)

print(f"Watermark cutoff   : {cutoff_s}")
print(f"Effective cutoff   : {effective_cutoff}")
print(f"Tương đương        : {datetime.utcfromtimestamp(effective_cutoff) if effective_cutoff else 'FULL LOAD'}")



# %%
# =============================================================================
# CELL 5 — ĐỌC BRONZE (1 LẦN), JOIN, ENRICH, PERSIST
# =============================================================================
# Giải thích:
#   SQL cũ đọc business_services 3 lần (1 lần / STEP).
#   Ở đây đọc 1 lần, join với partners + locations + dest_map + enrichment,
#   persist vào bộ nhớ → 3 transform dùng chung DataFrame này.
#
#   parse_amentities() gọi sau khi collect về Python per row
#   (trong transform functions ở Cell 6) — không gọi trong Spark UDF
#   vì UDF Python chậm, thay vào đó collect() rồi xử lý trong driver.
#   Với 450 records → collect() hoàn toàn an toàn về memory.
# =============================================================================

# Tính upper bound = max(svc.last_updated_at, i18n.updated_at)
upper_svc_row = (
    spark.table(BRZ_SERVICES)
    .filter(F.col("deleted") == False)
    .agg(F.max("last_updated_at").alias("m"))
    .first()
)
upper_i18n_row = (
    spark.table(BRZ_I18NS)
    .filter(F.col("deleted") == False)
    .agg(F.max("updated_at").alias("m"))
    .first()
)
upper_s = max(
    int(upper_svc_row["m"])  if upper_svc_row  and upper_svc_row["m"]  else 0,
    int(upper_i18n_row["m"]) if upper_i18n_row and upper_i18n_row["m"] else 0,
)
print(f"Upper bound (seconds): {upper_s}")
print(f"Tương đương          : {datetime.utcfromtimestamp(upper_s) if upper_s else 'N/A'}")

# --------------------------------------------------------------------------
# Đọc business_service_i18ns — pick up services có i18n thay đổi
# --------------------------------------------------------------------------
i18ns_df = (
    spark.table(BRZ_I18NS)
    .filter(F.col("deleted") == False)
    .select(
        F.col("business_service_id").alias("i18n_svc_id"),
        F.col("locale").alias("i18n_locale"),
        F.col("name").alias("i18n_name"),
        F.col("bio").alias("i18n_bio"),
        F.col("story").alias("i18n_story"),
        F.col("direction_note").alias("i18n_direction_note"),
        F.col("usps").alias("i18n_usps"),
        F.col("local_tips").alias("i18n_local_tips"),
        F.col("amentities").alias("i18n_amentities"),
        F.col("media").alias("i18n_media"),
        F.col("updated_at").alias("i18n_updated_at"),
    )
)

# Các service_id có i18n thay đổi từ sau cutoff
i18n_changed_ids = (
    i18ns_df
    .filter(F.col("i18n_updated_at") > effective_cutoff)
    .select("i18n_svc_id")
    .distinct()
)

# --------------------------------------------------------------------------
# Đọc bronze: records mới hơn watermark OR có i18n thay đổi
# --------------------------------------------------------------------------
svc_df = (
    spark.table(BRZ_SERVICES)
    .filter(F.col("deleted") == False)
    .join(i18n_changed_ids, F.col("id") == i18n_changed_ids["i18n_svc_id"], "left")
    .filter(
        (F.col("last_updated_at") > effective_cutoff)
        | F.col("i18n_svc_id").isNotNull()
    )
    .drop("i18n_svc_id")
)

svc_count = svc_df.count()
print(f"Services to process: {svc_count:,}")

if svc_count == 0:
    print("✓ Không có gì mới. Dừng lại.")
    raise SystemExit("NO_NEW_DATA")   # Fabric không có dbutils — dùng SystemExit

# --------------------------------------------------------------------------
# Join partners
# --------------------------------------------------------------------------
partners_df = (
    spark.table(BRZ_PARTNERS)
    .filter(F.col("deleted") == False)
    .select(
        F.col("id").alias("p_id"),
        F.col("type").alias("partner_type"),
        F.col("company_legal_name"),
        F.col("company_legal_vietnamese_name"),
        F.col("company_short_name"),
        F.col("last_updated_at").alias("p_updated_at"),
    )
)

locations_df = (
    spark.table(BRZ_LOCATIONS)
    .filter(F.col("deleted") == False)
    .select(
        F.col("id").alias("bl_id"),
        F.col("last_updated_at").alias("bl_updated_at"),
    )
)

# --------------------------------------------------------------------------
# Join destination mapping (ward → dest_name)
# --------------------------------------------------------------------------
dest_map_table = spark.table(GLD_DEST_MAP)
has_parent_dest_id = "parent_dest_id" in dest_map_table.columns

dest_map_df = (
    dest_map_table
    .filter(
        F.col("ward").isNotNull() & (F.trim(F.col("ward")) != "") &
        F.col("province_name").isNotNull() & (F.trim(F.col("province_name")) != "")
    )
    .withColumn("dm_ward", normalize_city_name("ward"))
    .withColumn("dm_province", normalize_city_name("province_name"))
    .select(
        "dm_ward",
        "dm_province",
        F.col("dest_name"),
        F.col("parent_dest_name"),
        F.col("dest_id"),
        F.col("parent_dest_id") if has_parent_dest_id else F.lit(None).cast("string").alias("parent_dest_id"),
    )
    .dropDuplicates(["dm_ward", "dm_province"])
)

# --------------------------------------------------------------------------
# Address enrichment (lat/lng từ Google Maps)
# --------------------------------------------------------------------------
enrich_df = (
    spark.table(GLD_POI_ADDR)
    .filter(F.col("google_lat").isNotNull())
    .select(
        F.col("poi_id").alias("enrich_poi_id"),
        F.col("google_lat").alias("lat"),
        F.col("google_lng").alias("lng"),
        F.col("vietmap_ward").alias("enriched_ward"),
        F.col("vietmap_district").alias("enriched_district"),
        F.col("vietmap_province").alias("enriched_province"),
    )
    # Deduplicate: giữ 1 record / poi_id
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("enrich_poi_id")
        .orderBy(F.col("lat").desc())
    ))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# --------------------------------------------------------------------------
# is_bookable từ ai_product
# --------------------------------------------------------------------------
bookable_df = (
    spark.table(GLD_AI_PRODUCT)
    .filter(F.col("is_bookable") == True)
    .select(F.col("source_id").alias("bk_source_id"))
    .distinct()
)

# --------------------------------------------------------------------------
# Join tất cả vào 1 DataFrame
# --------------------------------------------------------------------------
joined_df = (
    svc_df
    # Lưu partner_id của service trước khi join để tránh ambiguous sau join
    .withColumn("svc_partner_id", F.col("partner_id"))
    .join(partners_df,  svc_df["partner_id"] == partners_df["p_id"],  "left")
    .join(locations_df, svc_df["business_location_id"] == locations_df["bl_id"], "left")
    .join(bookable_df,  svc_df["id"] == bookable_df["bk_source_id"],  "left")
    # i18n: LEFT JOIN → 1 row per (svc, locale); service không có i18n → 1 row với NULL locale
    .join(i18ns_df, svc_df["id"] == i18ns_df["i18n_svc_id"], "left")
    .drop("i18n_svc_id")
    # lang: dùng i18n locale nếu có, fallback về 'vi'
    .withColumn("lang", F.coalesce(
        F.col("i18n_locale"),
        F.lit("vi"),
    ))
    # Override text fields bằng i18n nếu có
    .withColumn("name",           F.coalesce(F.col("i18n_name"),           F.col("name")))
    .withColumn("bio",            F.coalesce(F.col("i18n_bio"),            F.col("bio")))
    .withColumn("story",          F.coalesce(F.col("i18n_story"),          F.col("story")))
    .withColumn("direction_note", F.coalesce(F.col("i18n_direction_note"), F.col("direction_note")))
    .withColumn("amentities",     F.coalesce(F.col("i18n_amentities"),     F.col("amentities")))
    .withColumn("media",          F.coalesce(F.col("i18n_media"),          F.col("media")))
    .withColumn("usps",           F.coalesce(F.col("i18n_usps"),           F.col("usps")))
    .withColumn("local_tips",     F.coalesce(F.col("i18n_local_tips"),     F.col("local_tips")))
    .drop("i18n_locale", "i18n_name", "i18n_bio", "i18n_story",
          "i18n_direction_note", "i18n_amentities", "i18n_media", "i18n_updated_at",
          "i18n_usps", "i18n_local_tips")
    # Tính max_updated_at
    .withColumn("max_updated_at", F.greatest(
        F.coalesce("last_updated_at",    F.lit(0).cast("bigint")),
        F.coalesce("p_updated_at",       F.lit(0).cast("bigint")),
        F.coalesce("bl_updated_at",      F.lit(0).cast("bigint")),
    ))
    # Tính poi_id (deterministic) — encode lang để mỗi locale ra poi_id riêng
    .withColumn("poi_id", F.md5(F.concat(
        F.lit(INGESTION), F.col("id"), F.col("lang")
    )))
    # Address fields — [F3] full_address, [F4] NULLIF empty string
    .withColumn("addr_full",     F.get_json_object("address", "$.full_address"))
    .withColumn("addr_street",   F.get_json_object("address", "$.street"))
    .withColumn("addr_ward",     F.get_json_object("address", "$.ward"))
    .withColumn("addr_district", F.nullif(
        F.get_json_object("address", "$.district"), F.lit("")
    ))
    .withColumn("addr_city",     F.get_json_object("address", "$.city"))
    # is_bookable flag
    .withColumn("is_bookable",   F.col("bk_source_id").isNotNull())
)

# Tính poi_id đúng format UUID (dùng Python sau khi collect)
# Spark md5() trả về hex string, cần format lại ở Cell 6

# Join enrichment địa chỉ (lat/lng + enriched ward/district/province) TRƯỚC
joined_with_enrich = (
    joined_df
    .join(enrich_df, joined_df["poi_id"] == enrich_df["enrich_poi_id"], "left")
    .drop("enrich_poi_id")
    .withColumn("final_ward", F.coalesce(F.col("enriched_ward"), F.col("addr_ward")))
    .withColumn("final_province", F.coalesce(F.col("enriched_province"), F.col("addr_city")))
    .withColumn("_ward_norm", normalize_city_name("final_ward"))
    .withColumn("_prov_norm", normalize_city_name("final_province"))
)

# Join dest_map theo ward + city đã được lọc/chuẩn hóa qua normalize_city_name
final_df = (
    joined_with_enrich
    .join(dest_map_df,
          (F.col("_ward_norm") == dest_map_df["dm_ward"]) &
          (F.col("_prov_norm") == dest_map_df["dm_province"]),
          "left")
    .drop("_ward_norm", "_prov_norm", "dm_ward", "dm_province")
    # cast last_verify_at → string để tránh Spark parse timestamp rác (+193056-12-15)
    .withColumn("last_verify_at", F.col("last_verify_at").cast("string"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

total_rows = final_df.count()
print(f"✓ Joined rows: {total_rows:,}")


# %%
# =============================================================================
# CELL 6 — TRANSFORM FUNCTIONS
# =============================================================================
# Giải thích:
#   collect() kéo ~450 rows về Python driver (an toàn, nhỏ).
#   Mỗi row được xử lý bởi các hàm Python thuần — không UDF, không Spark.
#   Lợi ích:
#     - parse_json() gọi 1 lần / row thay vì 162 lần
#     - Logic phức tạp (is_24_7, venue_regulations) viết bằng Python dễ đọc
#     - Dễ unit test (Cell 10)
# =============================================================================

def _fmt_poi_id(hex_str: str) -> str:
    """Chuyển md5 hex string → UUID format 8-4-4-4-12."""
    h = hex_str.replace("-", "")
    return f"{h[:8]}-{h[8:12]}-{h[12:16]}-{h[16:20]}-{h[20:32]}"


def extract_operating_hours(am: dict, sector: str) -> dict:
    """
    Đọc operating_hours từ amentities dict.
    Trả về dict với keys: periods, is_24_7, holiday_exceptions

    [F7] Dùng open_time/close_time (đúng theo JSON thực tế)
    [F7] Tính is_24_7 thực sự thay vì hardcode False
    """
    cfg    = get_sector_cfg(sector)
    oh_raw = deep_get(am, *(cfg["oh_path"] or []))
    if not isinstance(oh_raw, dict):
        return {"periods": [], "is_24_7": False, "holiday_exceptions": []}

    DAY_MAP = {
        "monday": 1, "tuesday": 2, "wednesday": 3, "thursday": 4,
        "friday": 5, "saturday": 6, "sunday": 0,
    }

    periods        = []
    all_24h        = True
    has_any_open   = False

    for day_name, dow in DAY_MAP.items():
        day_data = oh_raw.get(day_name, {})
        if not isinstance(day_data, dict):
            all_24h = False
            continue

        status = str(day_data.get("status", "false")).lower() == "true"
        if not status:
            all_24h = False
            continue

        hours_list = day_data.get("hours", [])
        if not hours_list or not isinstance(hours_list, list):
            all_24h = False
            continue

        first = hours_list[0] if isinstance(hours_list[0], dict) else {}
        open_t  = (first.get("open_time")  or "").strip()
        close_t = (first.get("close_time") or "").strip()

        if not open_t:
            all_24h = False
            continue

        has_any_open = True

        # is_24_7: tất cả ngày open=00:00, close=00:00 hoặc 23:59
        is_this_day_24h = (open_t == "00:00" and close_t in ("00:00", "23:59", ""))
        if not is_this_day_24h:
            all_24h = False

        try:
            oh_parts = open_t.split(":")
            ch_parts = close_t.split(":") if close_t else ["0", "0"]
            periods.append({
                "day_of_week":  dow,
                "open_hour":    int(oh_parts[0]),
                "open_minute":  int(oh_parts[1]) if len(oh_parts) > 1 else 0,
                "close_hour":   int(ch_parts[0]),
                "close_minute": int(ch_parts[1]) if len(ch_parts) > 1 else 0,
            })
        except (ValueError, IndexError):
            all_24h = False

    holiday_ex = oh_raw.get("holiday_exceptions", [])

    return {
        "periods":            periods,
        "is_24_7":            all_24h and has_any_open,
        "holiday_exceptions": holiday_ex if isinstance(holiday_ex, list) else [],
    }


def extract_policy(am: dict, sector: str) -> dict:
    """
    [F8] Trích xuất policy đầy đủ:
      - cancellation_policy
      - check_in_out, breakfast, child, extra_bed (accommodation)
      - pet_policy per-sector
      - venue_regulations per-sector (trang phục, hút thuốc)
      - house_rules (f_n_b)
      - financial_transparency (spa)
      - terms_and_conditions
    """
    cfg      = get_sector_cfg(sector)
    policies = am.get("policies", {})

    cancel = deep_get(am, *(cfg["cancel_path"] or [])) if cfg["cancel_path"] else None

    # pet_policy per-sector
    if sector == "accommodation":
        pet = policies.get("pet_policy")
    elif sector == "f_n_b":
        pet = policies.get("house_rules", {})
        if isinstance(pet, dict):
            pet = pet.get("pet_allowed")
    else:
        pet = None

    # venue_regulations per-sector [F8]
    if sector == "attraction":
        venue_reg = {
            "gate_and_entry_rules": policies.get("gate_and_entry_rules"),
            "park_regulations":     policies.get("park_regulations"),
        }
    elif sector == "event":
        venue_reg = policies.get("venue_regulations")
    elif sector == "spa_n_wellness":
        venue_reg = policies.get("health_restrictions")
    else:
        venue_reg = None

    # house_rules: trang phục, hút thuốc (f_n_b)
    house_rules = policies.get("house_rules") if sector == "f_n_b" else None

    # financial_transparency: phí mang đồ ra ngoài, tipping (spa) [F8]
    financial_tr = (policies.get("financial_transparency")
                    if sector == "spa_n_wellness" else None)

    return {
        "cancellation_policy":    cancel,
        "check_in_out":           policies.get("check_in_out"),
        "breakfast_policy":       policies.get("breakfast_policy"),
        "child_policy":           policies.get("child_policy"),
        "extra_bed_policy":       policies.get("extra_bed_policy"),
        "pet_policy":             pet,
        "venue_regulations":      venue_reg,
        "house_rules":            house_rules,
        "financial_transparency": financial_tr,
        "terms_and_conditions":   am.get("terms_and_conditions"),
    }


def _to_enum(val: Any) -> Any:
    """Wrap string/list leaf values vào enum-schema; dict → recurse; bool/int/None → giữ nguyên."""
    if isinstance(val, list):
        return {"enum": val, "item_type": "tag", "required": "optional", "type": "array"}
    if isinstance(val, str):
        return {"enum": [val] if val else [], "item_type": "tag", "required": "optional", "type": "string"}
    if isinstance(val, dict):
        return {k: _to_enum(v) for k, v in val.items()}
    return val


def extract_amenities_json(am: dict, sector: str) -> str:
    """
    [F6] Build amenities_json cho poi_detail.
    Drop 'policies' (đã có poi_policy_json riêng).
    Giữ cấu trúc sector gốc (center_profile, logistics, facilities, v.v.)
    và wrap mỗi string/list leaf value vào enum-schema.
    """
    result = {k: _to_enum(v) for k, v in am.items() if k != "policies"}
    return json.dumps(result, ensure_ascii=False) if result else "{}"


def build_operating_status(row: dict) -> str:
    """Tính operating_status từ is_active + operation_status."""
    if not row.get("is_active") and row.get("last_in_activated_at"):
        return "closed_temporary"
    op = clean(row.get("operation_status"))
    return op or "operational"


def build_row_base(row: dict, am: dict) -> dict:
    """
    Xây dựng các trường dùng chung cho cả 3 bảng.
    Gọi 1 lần / row, kết quả dùng lại cho poi_master, ai_poi, poi_detail.
    """
    svc_id    = row["id"]
    sector    = row.get("business_sector") or ""
    poi_id    = _fmt_poi_id(row["poi_id"])  # format UUID
    name      = clean(row.get("name")) or ""
    cfg       = get_sector_cfg(sector)

    # Category
    cat_raw = deep_get(am, *(cfg["category_path"] or [])) if cfg["category_path"] else None
    if isinstance(cat_raw, list):
        category = clean(cat_raw[0]) if cat_raw else None
    else:
        category = clean(cat_raw)
    if category is None and sector == "other":
        category = clean(row.get("business_sector"))

    # Address [F3][F4]
    addr_full     = clean(row.get("addr_full"))
    addr_street   = clean(row.get("addr_street"))
    addr_ward     = nullif_empty(clean(row.get("addr_ward")))
    addr_district = nullif_empty(clean(row.get("addr_district")))
    addr_city     = nullif_empty(clean(row.get("addr_city")))

    # Use enriched values if available
    enriched_ward     = clean(row.get("enriched_ward"))
    enriched_district = clean(row.get("enriched_district"))
    enriched_province = clean(row.get("enriched_province"))

    ward_final     = enriched_ward     or addr_ward
    district_final = enriched_district or addr_district  # [F4]
    city_final     = enriched_province or addr_city

    # Destination
    dest_name        = clean(row.get("dest_name"))
    parent_dest_name = clean(row.get("parent_dest_name"))
    dest_id          = clean(row.get("dest_id"))
    parent_dest_id   = clean(row.get("parent_dest_id"))

    dest_ids = []
    dest_names = []

    if dest_id:
        dest_ids.append(dest_id)
    if parent_dest_id and parent_dest_id != dest_id:
        dest_ids.append(parent_dest_id)

    if dest_name:
        dest_names.append(dest_name)
    if parent_dest_name and parent_dest_name != dest_name:
        dest_names.append(parent_dest_name)

    # Timestamps
    created_at    = epoch_ms_to_ts(row.get("created_at"))
    max_updated   = max(
        row.get("last_updated_at") or 0,
        row.get("p_updated_at")    or 0,
        row.get("bl_updated_at")   or 0,
    )
    updated_at = epoch_ms_to_ts(max_updated)

    # Operating hours [F7]
    oh_data = extract_operating_hours(am, sector)

    # Policy [F8]
    policy_data = extract_policy(am, sector)

    return {
        "poi_id":         poi_id,
        "svc_id":         svc_id,
        "lang":           row.get("lang") or "vi",
        "slug":           make_slug(name, poi_id),
        "poi_name":       name,
        "sector":         sector,
        "category":       category,
        "ingestion":      INGESTION,
        "is_active":      bool(row.get("is_active")),
        # last_verify_at đã cast thành string — None/"None"/"null"/rác = chưa verify
        "is_verified":    bool(
            row.get("last_verify_at") and
            str(row.get("last_verify_at")).strip() not in ("None", "null", "", "nan")
        ),
        "is_bookable":    bool(row.get("is_bookable")),
        "operating_status": build_operating_status(row),
        "partner_id":     row.get("svc_partner_id"),    # [F5] dùng svc_partner_id tránh ambiguous sau join
        "partner_type":   row.get("partner_type"),
        "company_legal":  clean(row.get("company_legal_name") or row.get("company_legal_vietnamese_name")),
        "company_short":  clean(row.get("company_short_name")),
        "bio":            clean(row.get("bio")),
        "story":          clean(row.get("story")),
        "usps":           clean(row.get("usps")),
        "local_tips":     clean(row.get("local_tips")),
        "addr_full":      addr_full,
        "addr_street":    addr_street,
        "addr_ward":      ward_final,
        "addr_district":  district_final,
        "addr_city":      city_final,
        "lat":            row.get("lat"),
        "lng":            row.get("lng"),
        "phone":          clean(row.get("phone")),
        "hotline":        clean(row.get("hotline")),
        "email":          clean(row.get("email")),
        "website_url":    clean(
            (parse_json(row.get("website_urls")) or {}).get("website_url")
        ),
        "dest_ids":       json.dumps(dest_ids, ensure_ascii=False) if dest_ids else None,
        "dest_id":        dest_id or dest_name,
        "dest_name":      dest_name,
        "dest_names":     json.dumps(dest_names, ensure_ascii=False) if dest_names else None,
        "created_at":     created_at,
        "updated_at":     updated_at,
        "oh":             oh_data,
        "policy":         policy_data,
        "am":             am,           # full amentities dict, dùng lại cho poi_detail
    }


# --------------------------------------------------------------------------
# Amenity matching helpers — Confluence "5.2 Map các cột amenity theo từng
# business_sector" (v1.5, 21/7). Match GIÁ TRỊ CHÍNH XÁC theo đúng JSONPath
# của từng sector — không dùng "contains" từ khoá đa ngôn ngữ như bản cũ.
# --------------------------------------------------------------------------

def _as_list(v: Any) -> list:
    if v is None:
        return []
    return v if isinstance(v, list) else [v]


def path_has(am: dict, path: list, values: list) -> bool:
    """True nếu mảng (hoặc scalar) tại path chứa đúng 1 trong các giá trị
    (so khớp chính xác, không phân biệt hoa/thường)."""
    items = _as_list(deep_get(am, *path))
    wanted = {v.lower() for v in values}
    return any(isinstance(x, str) and x.strip().lower() in wanted for x in items)


def path_is_true(am: dict, path: list) -> bool:
    """True nếu giá trị tại path là boolean true (hoặc chuỗi 'true')."""
    v = deep_get(am, *path)
    return v if isinstance(v, bool) else str(v).strip().lower() == "true"


def path_exists(am: dict, path: list) -> bool:
    """True nếu giá trị tại path tồn tại và không rỗng."""
    v = deep_get(am, *path)
    if v is None:
        return False
    return v.strip() != "" if isinstance(v, str) else True


def path_positive(am: dict, path: list) -> bool:
    """True nếu giá trị tại path là số > 0."""
    v = deep_get(am, *path)
    try:
        return v is not None and float(v) > 0
    except (TypeError, ValueError):
        return False


def path_equals_number(am: dict, path: list, target: float) -> bool:
    """True nếu giá trị tại path là số và bằng target."""
    v = deep_get(am, *path)
    try:
        return v is not None and float(v) == float(target)
    except (TypeError, ValueError):
        return False


def path_list_has_key_equal(am: dict, path: list, key: str, target: float) -> bool:
    """True nếu mảng tại path có ít nhất 1 phần tử mà item[key] == target."""
    arr = deep_get(am, *path)
    if not isinstance(arr, list):
        return False
    for item in arr:
        if not isinstance(item, dict):
            continue
        try:
            if float(item.get(key)) == float(target):
                return True
        except (TypeError, ValueError):
            continue
    return False


def menu_has(am: dict, field: str, values: list) -> bool:
    """True nếu 1 phần tử trong am['menu'] có field chứa 1 trong các giá trị."""
    menu = am.get("menu")
    if not isinstance(menu, list):
        return False
    wanted = {v.lower() for v in values}
    for item in menu:
        if not isinstance(item, dict):
            continue
        if any(isinstance(x, str) and x.strip().lower() in wanted for x in _as_list(item.get(field))):
            return True
    return False


# Mỗi cột ai_poi → {sector: rule(am) -> bool}. Sector không có trong dict = mặc định False.
AMENITY_RULES = {
    "wifi_free": {
        "accommodation":        lambda am: path_has(am, ["facilities", "general_amenities"], ["Free Wi-Fi"]),
        "attraction":           lambda am: path_has(am, ["facilities", "facilities_and_services"], ["Free Wi-Fi"]),
        "shopping_n_retail":    lambda am: path_has(am, ["store_profile", "facilities_and_services"], ["Free Wi-Fi"]),
        "tour_activities":      lambda am: path_has(am, ["logistics", "fleet_amenities"], ["On-board Wi-Fi"]),
        "transport_route_base": lambda am: path_has(am, ["fleet_profile", "on_board_amenities"], ["Free Wi-Fi"]),
    },
    "has_swimming_pool": {
        "accommodation": lambda am: path_has(am, ["facilities", "general_amenities"], ["Swimming Pool"]),
    },
    "has_gym": {
        "accommodation": lambda am: path_has(am, ["facilities", "general_amenities"], ["Gym"]),
    },
    "has_spa": {
        "accommodation":  lambda am: path_has(am, ["facilities", "general_amenities"], ["Spa & Wellness"]),
        "spa_n_wellness": lambda am: True,
    },
    "has_bar_on_site": {
        "accommodation": lambda am: path_has(am, ["facilities", "general_amenities"], ["Bar"]),
        "f_n_b":          lambda am: (
            path_has(am, ["facilities", "main_cuisine"], ["Bar & Pub/ Nightlife"]) or
            path_has(am, ["facilities", "specialty_tags"], ["Bar-Pub", "Rooftop Bar"])
        ),
    },
    "has_restaurant_on_site": {
        "accommodation": lambda am: path_has(am, ["facilities", "general_amenities"], ["Restaurant"]),
        "attraction":     lambda am: path_has(am, ["facilities", "facilities_and_services"], ["Restaurant-Cafe"]),
    },
    "has_garden": {
        "accommodation": lambda am: path_has(am, ["facilities", "general_amenities"], ["Garden"]),
    },
    "has_sauna": {
        "accommodation": lambda am: path_has(am, ["facilities", "general_amenities"], ["Sauna"]),
    },
    "has_jacuzzi": {
        "accommodation": lambda am: path_has(am, ["facilities", "general_amenities"], ["Jacuzzi"]),
    },
    "has_meeting_rooms": {
        "accommodation": lambda am: path_has(am, ["facilities", "general_amenities"], ["Meeting Rooms"]),
    },
    "has_coworking_space": {
        "accommodation": lambda am: path_has(am, ["facilities", "general_amenities"], ["Co-working Space"]),
    },
    "has_airport_shuttle": {
        "accommodation": lambda am: path_has(am, ["facilities", "general_amenities"], ["Airport Shuttle"]),
    },
    "front_desk_24h": {
        "accommodation": lambda am: path_has(am, ["facilities", "general_amenities"], ["24-hour Front Desk"]),
    },
    "luggage_storage": {
        "accommodation": lambda am: path_has(am, ["facilities", "general_amenities"], ["Luggage Storage"]),
        "attraction":     lambda am: path_has(am, ["facilities", "facilities_and_services"], ["Lockers"]),
    },
    "currency_exchange": {
        "accommodation": lambda am: path_has(am, ["facilities", "general_amenities"], ["Currency Exchange"]),
    },
    "ev_charge": {
        "accommodation": lambda am: path_has(am, ["facilities", "general_amenities"], ["EV Charging Station"]),
    },
    "braille_elevator": {
        "accommodation": lambda am: path_has(am, ["facilities", "accessibility"], ["Braille Elevator"]),
    },
    "lowered_sink": {
        "accommodation": lambda am: path_has(am, ["facilities", "accessibility"], ["Lowered Sink"]),
    },
    "air_conditioning": {
        "spa_n_wellness":       lambda am: path_has(am, ["center_profile", "facilities_and_ambience"], ["Air-conditioned"]),
        "tour_activities":      lambda am: path_has(am, ["logistics", "fleet_amenities"], ["Air-Conditioned"]),
        "transport_route_base": lambda am: path_has(am, ["fleet_profile", "on_board_amenities"], ["Air Conditioning"]),
    },
    "has_restroom": {
        "accommodation":        lambda am: True,
        "transport_route_base": lambda am: path_has(am, ["fleet_profile", "on_board_amenities"], ["On-board Toilet"]),
    },
    "accessibility_options_wheelchair_accessible_parking": {
        "accommodation": lambda am: path_has(am, ["facilities", "accessibility"], ["Accessible Parking"]),
    },
    "accessibility_options_wheelchair_accessible_entrance": {
        "accommodation": lambda am: path_has(am, ["facilities", "accessibility"], ["Wheelchair Path"]),
    },
    "accessibility_options_wheelchair_accessible_restroom": {
        "accommodation": lambda am: path_has(am, ["facilities", "accessibility"], ["Grab Rail Toilet"]),
    },
    "parking": {
        "accommodation":     lambda am: path_has(am, ["facilities", "general_amenities"], ["Parking"]),
        "attraction":        lambda am: path_has(am, ["facilities", "facilities_and_services"], ["Parking"]),
        "f_n_b":             lambda am: path_has(am, ["facilities", "space_and_services"], ["Car Parking"]),
        "shopping_n_retail": lambda am: path_has(am, ["store_profile", "facilities_and_services"], ["Parking"]),
        "spa_n_wellness":    lambda am: path_has(am, ["center_profile", "facilities_and_ambience"], ["Car Parking Available"]),
    },
    "payment_options_accepts_credit_cards": {
        "shopping_n_retail": lambda am: (
            path_has(am, ["store_profile", "facilities_and_services"], ["Credit Card (Visa/Master)"]) or
            path_has(am, ["policies", "fulfillment_and_payment", "supported_payments"], ["Visa/Mastercard"])
        ),
    },
    "payment_options_accepts_debit_cards": {
        "shopping_n_retail": lambda am: path_has(am, ["policies", "fulfillment_and_payment", "supported_payments"], ["Napas"]),
    },
    "payment_options_accepts_nfc": {
        "shopping_n_retail": lambda am: path_has(am, ["policies", "fulfillment_and_payment", "supported_payments"], ["Apple Pay"]),
    },
    "outdoor_seating": {
        "f_n_b":          lambda am: path_has(am, ["facilities", "space_and_services"], ["Outdoor Seating"]),
        "spa_n_wellness": lambda am: path_has(am, ["center_profile", "facilities_and_ambience"], ["Outdoor/Al Fresco Seating"]),
    },
    "live_music": {
        "event":          lambda am: path_has(am, ["organizer_profile", "specialty_topics"], ["Concert-Live Music"]),
        "f_n_b":          lambda am: path_has(am, ["facilities", "space_and_services"], ["Live Music"]),
        "spa_n_wellness": lambda am: path_has(am, ["center_profile", "facilities_and_ambience"], ["Live Music"]),
    },
    "takeout": {
        "f_n_b": lambda am: menu_has(am, "operational_tags", ["Takeaway/Delivery valid"]),
    },
    "delivery": {
        "f_n_b": lambda am: (
            path_has(am, ["facilities", "space_and_services"], ["Delivery"]) or
            menu_has(am, "operational_tags", ["Takeaway/Delivery valid"])
        ),
        "shopping_n_retail": lambda am: path_has(
            am, ["policies", "fulfillment_and_payment", "shipping_coverage"],
            ["Local/City Delivery", "Nationwide Shipping", "International Shipping"],
        ),
    },
    "dine_in": {
        "f_n_b": lambda am: path_has(am, ["facilities", "space_and_services"], ["Table Service"]),
    },
    "reservable": {
        "event":          lambda am: path_is_true(am, ["policies", "refund_and_postponement", "status"]),
        "f_n_b":          lambda am: path_is_true(am, ["policies", "deposit_and_cancellation", "status"]),
        "spa_n_wellness": lambda am: path_is_true(am, ["policies", "financial_transparency", "native_booking"]),
    },
    "serves_breakfast": {
        "accommodation": lambda am: path_is_true(am, ["policies", "breakfast_policy"]),
    },
    "serves_lunch": {
        "f_n_b": lambda am: path_has(am, ["facilities", "specialty_tags"], ["Office Lunch"]),
    },
    "serves_beer": {
        "f_n_b": lambda am: menu_has(am, "dietary_and_allergy", ["Alcohol/18+"]),
    },
    "serves_wine": {
        "f_n_b": lambda am: menu_has(am, "dietary_and_allergy", ["Alcohol/18+"]),
    },
    "serves_vegetarian_food": {
        "f_n_b": lambda am: (
            path_has(am, ["facilities", "main_cuisine"], ["Vegan Restaurant"]) or
            menu_has(am, "dietary_and_allergy", ["Vegetarian", "Vegan"])
        ),
    },
    "serves_dessert": {
        "f_n_b": lambda am: path_has(am, ["facilities", "main_cuisine"], ["Bakery & Dessert"]),
    },
    "serves_coffee": {
        "f_n_b": lambda am: path_has(am, ["facilities", "specialty_tags"], ["Coffee Shop"]),
    },
    "menu_for_children": {
        "f_n_b": lambda am: (
            path_is_true(am, ["policies", "child_policy", "child_menu_available"]) or
            menu_has(am, "dietary_and_allergy", ["Kid-Friendly"])
        ),
    },
    "good_for_children": {
        "accommodation":        lambda am: path_is_true(am, ["policies", "child_policy", "status"]),
        "f_n_b":                lambda am: path_is_true(am, ["policies", "child_policy", "status"]),
        "spa_n_wellness":       lambda am: path_equals_number(am, ["policies", "health_restrictions", "min_age"], 0),
        "transport_route_base": lambda am: path_exists(am, ["policies", "luggage_and_child_policy", "free_infant_height_cm"]),
    },
    "allows_dogs": {
        "accommodation": lambda am: path_is_true(am, ["policies", "pet_policy"]),
        "f_n_b":          lambda am: path_is_true(am, ["policies", "house_rules", "pet_allowed"]),
    },
    "good_for_groups": {
        "event": lambda am: True,
        "f_n_b": lambda am: menu_has(am, "commercial_category", ["Combo/Platter", "Buffet Ticket/Pass"]),
    },
    "good_for_watching_sports": {
        "f_n_b": lambda am: path_has(am, ["facilities", "space_and_services"], ["Sports Screening"]),
    },
    "pure_service_area_business": {
        "transport_on_demand":  lambda am: True,
        "transport_route_base": lambda am: True,
    },
    "free_cancellation": {
        "accommodation": lambda am: path_has(am, ["policies", "cancellation_policy", "policy"],
                                              ["Standard Flexible", "Moderate", "Strict"]),
        "attraction":    lambda am: path_is_true(am, ["policies", "refund_and_change_policy", "cancellation_allowed"]),
        "event":         lambda am: path_is_true(am, ["policies", "refund_and_postponement", "refund_allowed"]),
        "f_n_b":         lambda am: path_exists(am, ["policies", "deposit_and_cancellation", "cancellation_window_hours"]),
        "shopping_n_retail": lambda am: path_is_true(am, ["policies", "return_and_exchange", "status"]),
        "spa_n_wellness":    lambda am: path_positive(am, ["policies", "late_arrival_and_cancellation", "free_cancellation_hours"]),
        "tour_activities": lambda am: path_list_has_key_equal(am, ["policies", "cancellation_policy", "rules"], "penalty_percentage", 0),
        "transport_on_demand":  lambda am: path_positive(am, ["policies", "cancellation_policy", "free_cancellation_hours"]),
        "transport_route_base": lambda am: path_is_true(am, ["policies", "cancellation_policy", "cancellation_allowed"]),
        "vehicle_rental":       lambda am: path_positive(am, ["policies", "cancellation_policy", "free_cancellation_hours"]),
    },
}


def build_amenities_flags(am: dict, sector: str) -> dict:
    """
    Tính cờ boolean cho ai_poi theo bảng mapping amenity (Confluence v1.5, mục 5.2).
    Mặc định False nếu sector không có rule cho cột đó (đúng theo ghi chú spec).
    """
    flags = {}
    for col, sector_rules in AMENITY_RULES.items():
        rule = sector_rules.get(sector)
        flags[col] = bool(rule(am)) if rule else False
    return flags


def build_general_amenities(flags: dict) -> list:
    """Cột general_amenities (JSON) = tên các cột amenity có giá trị True."""
    return [col for col, val in flags.items() if val]


def build_media(media_raw: str | None) -> dict:
    """Parse media JSON array → cover_url + gallery_urls."""
    items = []
    s = clean(media_raw)
    if s:
        try:
            parsed = json.loads(s)
            if isinstance(parsed, list):
                items = parsed
        except (json.JSONDecodeError, TypeError):
            pass
    cover  = next((x.get("url") for x in items if isinstance(x, dict) and x.get("is_cover")), None)
    others = sorted(
        [x.get("url") for x in items if isinstance(x, dict) and not x.get("is_cover") and x.get("url")],
        key=lambda _: 0
    )
    return {
        "cover_url":   cover,
        "gallery_urls": json.dumps(others),
        "media_json":  json.dumps({
            "cover":       {"media_id": str(uuid.uuid4()), "blob_url": cover} if cover else None,
            "gallery":     [{"media_id": str(uuid.uuid4()), "blob_url": u} for u in others],
            "total_count": len(items),
        }),
    }


print("✓ Cell 6: transform functions ready")


# %%
# =============================================================================
# CELL 7 — BUILD DATAFRAMES (collect → transform → createDataFrame)
# =============================================================================
# Giải thích:
#   collect() kéo ~450 rows về driver Python.
#   Mỗi row được xử lý bởi các hàm ở Cell 6.
#   Kết quả gom vào 3 list: master_rows, ai_poi_rows, detail_rows.
#   Sau đó spark.createDataFrame() tạo 3 DataFrame để MERGE.
#
#   Tại sao collect() thay vì UDF?
#   - 450 rows nhỏ, collect an toàn
#   - Python logic thuần, không cần serialize/deserialize UDF overhead
#   - Dễ debug: print(row) trực tiếp
# =============================================================================

rows = final_df.collect()
print(f"Collected {len(rows):,} rows")

master_rows  = []
ai_poi_rows  = []
detail_rows  = []
error_count  = 0

for raw_row in rows:
    row = raw_row.asDict()
    try:
        am       = parse_json(row.get("amentities"))   # parse 1 lần / record
        base     = build_row_base(row, am)
        media    = build_media(row.get("media"))
        flags    = build_amenities_flags(am, base["sector"])
        oh       = base["oh"]
        policy   = base["policy"]

        poi_id   = base["poi_id"]
        sector   = base["sector"]

        # ----------------------------------------------------------------
        # poi_master row
        # ----------------------------------------------------------------
        master_rows.append((
            poi_id,                         # poi_id
            base["lang"],                   # lang
            base["slug"],                   # slug
            base["poi_name"],               # poi_name
            sector,                         # business_sector   [F1]
            base["category"],               # business_category
            None,                           # subcategory_tags
            base["operating_status"],       # operating_status
            base["is_active"],              # is_active
            base["is_verified"],            # is_verified
            base["bio"],                    # short_description
            base["addr_full"],              # short_address     [F3]
            base["addr_district"],          # district          [F4]
            None,                           # rating_overall
            0,                              # rating_count
            base["dest_ids"],               # destination_ids
            None,                           # nearby_pois
            base["dest_names"],             # destination_names
            base["created_at"],             # created_at
            base["updated_at"],             # updated_at
            0,                              # version
            base["svc_id"],                 # source_id
            None,                           # event_id
            None,                           # ref_type
            None,                           # ref_business_sector
            INGESTION,                      # ingestion_source
            base["partner_id"],             # partner_id        [F5]
            base["is_bookable"],            # is_bookable
            True,                           # is_recommended  
        ))

        # ----------------------------------------------------------------
        # ai_poi row
        # ----------------------------------------------------------------
        # Build operating_hours JSON string
        oh_periods = oh["periods"]
        oh_json = json.dumps({
            "timezone":                "Asia/Ho_Chi_Minh",
            "is_24_7":                 oh["is_24_7"],      # [F7]
            "periods_json":            oh_periods,
            "holiday_exceptions_json": oh["holiday_exceptions"],
        }, ensure_ascii=False)

        cancel_json = json.dumps(policy["cancellation_policy"], ensure_ascii=False) if policy["cancellation_policy"] else None

        ai_poi_rows.append((
            poi_id,                                          # poi_id
            base["lang"],                                    # lang
            base["slug"],                                    # slug
            base["poi_name"],                                # poi_name
            base["company_legal"],                           # alternative_name
            sector,                                          # business_sector
            base["category"],                                # business_category
            None,                                            # subcategory_tags
            INGESTION,                                       # ingestion_source
            base["partner_type"],                            # poi_types
            base["dest_id"],                                 # destination_id
            base["dest_name"],                               # destination_name
            base["is_active"],                               # is_active
            base["is_verified"],                             # is_verified
            base["company_short"] is not None,               # brand_or_chain
            base["addr_street"] or base["addr_full"],        # short_address
            base["addr_full"],                               # address_line_1
            None,                                            # address_line_2
            base["addr_ward"],                               # ward
            base["addr_district"],                           # district
            base["addr_city"],                               # province_name
            None,                                            # country_code
            base["lat"],                                     # lat
            base["lng"],                                     # lng
            base["phone"],                                   # phone
            base["website_url"],                             # website_url
            None,                                            # rating_overall
            0,                                               # rating_count
            None,                                            # price_level
            None,                                            # price_min
            None,                                            # price_max
            media["cover_url"],                              # cover_thumbnail_url
            media["cover_url"],                              # cover_original_url
            media["gallery_urls"],                           # gallery_urls
            base["story"],                                   # story
            base["bio"],                                     # description
            base["usps"],                                    # usp_highlights
            base["local_tips"],                              # local_tips_or_neighborhood
            oh_json,                                         # operating_hours
            oh["is_24_7"],                                   # is_24_7
            json.dumps(oh["holiday_exceptions"], ensure_ascii=False),            # holiday_exceptions
            cancel_json,                                     # cancellation_policy
            flags["takeout"],                                # takeout
            flags["delivery"],                                # delivery
            flags["dine_in"],                                # dine_in
            False,                                           # curbside_pickup [không có data từ partner portal]
            flags["reservable"],                             # reservable
            flags["serves_breakfast"],                       # serves_breakfast
            flags["serves_lunch"],                           # serves_lunch
            False,                                           # serves_dinner   [không có data từ partner portal]
            flags["serves_beer"],                            # serves_beer
            flags["serves_wine"],                            # serves_wine
            False,                                           # serves_brunch   [không có data từ partner portal]
            flags["serves_vegetarian_food"],                 # serves_vegetarian_food
            flags["outdoor_seating"],                        # outdoor_seating
            flags["live_music"],                             # live_music
            flags["menu_for_children"],                      # menu_for_children
            None,                                            # serves_cocktails [chưa có mapping trong spec]
            flags["serves_dessert"],                         # serves_dessert
            flags["serves_coffee"],                          # serves_coffee
            flags["good_for_children"],                      # good_for_children
            flags["allows_dogs"],                            # allows_dogs
            flags["has_restroom"],                           # has_restroom
            flags["good_for_groups"],                        # good_for_groups
            flags["good_for_watching_sports"],               # good_for_watching_sports
            flags["pure_service_area_business"],             # pure_service_area_business
            flags["parking"],                                # parking
            flags["ev_charge"],                              # ev_charge
            flags["payment_options_accepts_credit_cards"],   # payment_options_accepts_credit_cards
            flags["payment_options_accepts_debit_cards"],    # payment_options_accepts_debit_cards
            flags["payment_options_accepts_nfc"],            # payment_options_accepts_nfc
            False,                                           # parking_options_free_parking_lot
            False,                                           # parking_options_paid_parking_lot
            False,                                           # parking_options_free_street_parking
            False,                                           # parking_options_paid_street_parking
            False,                                           # parking_options_valet_parking
            False,                                           # parking_options_free_garage_parking
            False,                                           # parking_options_paid_garage_parking
            flags["accessibility_options_wheelchair_accessible_parking"],
            flags["accessibility_options_wheelchair_accessible_entrance"],
            flags["accessibility_options_wheelchair_accessible_restroom"],
            False,                                           # accessibility_options_wheelchair_accessible_seating [không có data]
            json.dumps(build_general_amenities(flags), ensure_ascii=False),  # general_amenities
            base["created_at"],                              # created_at
            base["updated_at"],                              # updated_at
            base["svc_id"],                                  # source_id
            None,                                            # event_id
            flags["has_swimming_pool"],                      # has_swimming_pool
            flags["has_gym"],                                # has_gym
            flags["has_spa"],                                # has_spa
            flags["has_sauna"],                              # has_sauna
            flags["has_garden"],                             # has_garden
            flags["has_bar_on_site"],                        # has_bar_on_site
            flags["has_restaurant_on_site"],                 # has_restaurant_on_site
            flags["has_coworking_space"],                    # has_coworking_space
            base["company_short"],                           # brand_name
            flags["wifi_free"],                              # wifi_free
            flags["has_jacuzzi"],                            # has_jacuzzi
            flags["has_meeting_rooms"],                      # has_meeting_rooms
            flags["has_airport_shuttle"],                    # has_airport_shuttle
            flags["front_desk_24h"],                         # front_desk_24h
            flags["luggage_storage"],                        # luggage_storage
            flags["currency_exchange"],                      # currency_exchange
            flags["air_conditioning"],                       # air_conditioning
            flags["free_cancellation"],                      # free_cancellation
            flags["lowered_sink"],                           # lowered_sink
            flags["braille_elevator"],                       # braille_elevator
        ))

        # ----------------------------------------------------------------
        # poi_detail row
        # ----------------------------------------------------------------
        amenities_json = extract_amenities_json(am, sector)  # [F6]

        address_json = json.dumps({
            "address_line1":  row.get("addr_street") or "",
            "address_line2":  "",
            "ward":           base["addr_ward"],
            "district":       base["addr_district"],    # [F4]
            "city":           base["addr_city"],
            "state_province": base["addr_city"],
            "country_code":   "VN",
            "postal_code":    None,
            "short_address":  base["addr_street"],
            "full_address":   base["addr_full"],        # [F3]
            "lat":            base["lat"],
            "lng":            base["lng"],
            "plus_code":      None,
        }, ensure_ascii=False)

        contact_json = json.dumps({
            "phone":       base["phone"],
            "hotline":     base["hotline"],
            "email":       base["email"],
            "website_url": base["website_url"],
        }, ensure_ascii=False)

        content_json = json.dumps({
            "vi": {
                "display_name":    base["poi_name"],
                "description_short": base["bio"],
                "description_long":  base["story"],
                "service_bio":       base["bio"],
            }
        }, ensure_ascii=False)

        policy_json = json.dumps({
            "policy": {
                "cancellation_policy":    policy["cancellation_policy"],
                "check_in_out":           policy["check_in_out"],
                "breakfast_policy":       policy["breakfast_policy"],
                "child_policy":           policy["child_policy"],
                "extra_bed_policy":       policy["extra_bed_policy"],
                "pet_policy":             policy["pet_policy"],       # [F8]
                "venue_regulations":      policy["venue_regulations"], # [F8]
                "house_rules":            policy["house_rules"],       # [F8]
                "financial_transparency": policy["financial_transparency"], # [F8]
                "terms_and_conditions":   policy["terms_and_conditions"],
            },
            "source_updated_at": str(base["updated_at"]),
        }, ensure_ascii=False)

        opening_hours_json = json.dumps({
            "timezone":                "Asia/Ho_Chi_Minh",
            "is_24_7":                 oh["is_24_7"],      # [F7]
            "periods_json":            oh["periods"],
            "holiday_exceptions_json": oh["holiday_exceptions"],
        }, ensure_ascii=False)

        extra_json = json.dumps({
            "business_sector":      sector,
            "operation_status":     base["operating_status"],
            "is_main":              bool(row.get("is_main")),
            "metadata":             parse_json(row.get("metadata")),
            "partner_id":           base["partner_id"],
            "business_location_id": row.get("business_location_id"),
        }, ensure_ascii=False)

        detail_rows.append((
            poi_id,             # poi_id
            base["lang"],       # lang
            amenities_json,     # amenities_json
            address_json,       # poi_address_json
            contact_json,       # poi_contact_json
            content_json,       # poi_content_json
            media["media_json"], # poi_media_json
            opening_hours_json, # poi_opening_hours_json
            policy_json,        # poi_policy_json
            "{}",               # poi_rating_json
            "[]",               # poi_review_json
            extra_json,         # poi_extra_json
            base["svc_id"],     # source_id
            md5(f"{base['svc_id']}{RUN_ID}".encode()).hexdigest(),  # event_id (deterministic)
            INGESTION,          # ingestion_source
            base["created_at"], # created_at
            base["updated_at"], # updated_at
        ))

    except Exception as e:
        error_count += 1
        print(f"❌ ERROR svc_id={row.get('id')}: {e}")
        if error_count > 10:
            raise RuntimeError("Quá nhiều lỗi, dừng lại") from e

print(f"✓ Built {len(master_rows):,} poi_master rows")
print(f"✓ Built {len(ai_poi_rows):,} ai_poi rows")
print(f"✓ Built {len(detail_rows):,} poi_detail rows")
print(f"⚠ Errors: {error_count}")


# %%
# =============================================================================
# CELL 8 — MERGE 3 BẢNG GOLD
# =============================================================================

# %%
# =============================================================================
# CELL 8 — MERGE 3 BẢNG GOLD
# =============================================================================
#
# DESIGN:
#   - KHÔNG dùng whenNotMatchedInsertAll()
#   - Chỉ INSERT các column tồn tại ở cả source và target.
#   - Target có thêm column mới sẽ KHÔNG làm pipeline fail nếu source chưa có.
#
# Ownership:
#   Partner Portal quản lý các field được build trong master_df.
#
#   Các field enrichment / business khác:
#       poi_group_id
#       is_recommended
#
#   => Không update từ Partner Portal.
#   => Insert POI mới: để NULL.
#
# Lợi ích:
#   1. Fix lỗi:
#      DELTA_MERGE_UNRESOLVED_EXPRESSION poi_group_id
#
#   2. Không overwrite poi_group_id/is_recommended hiện có.
#
#   3. Future-proof:
#      target thêm column mới không làm MERGE INSERT fail.
#
# =============================================================================


# =============================================================================
# COMMON HELPERS
# =============================================================================

def to_spark(rows: list, schema: StructType) -> DataFrame:
    """
    Tạo DataFrame từ list of tuples với schema rõ ràng.
    """
    return spark.createDataFrame(rows, schema=schema)


def build_safe_insert_map(
    source_df: DataFrame,
    target_table: str,
    exclude_columns: set | None = None,
) -> dict:
    """
    Build mapping cho WHEN NOT MATCHED INSERT.

    Chỉ insert những column:
        - tồn tại trong source DataFrame
        - tồn tại trong target Delta table
        - không nằm trong exclude_columns

    Mục đích:
        - Không dùng InsertAll().
        - Target thêm column mới không làm pipeline fail.
        - Column target không có trong source sẽ nhận NULL/default
          theo schema của target.

    Ví dụ:
        target có:
            poi_group_id
            is_recommended

        source không có:
            poi_group_id
            is_recommended

        => Hai column này KHÔNG xuất hiện trong insert map
        => Delta insert NULL/default.
    """

    exclude_columns = exclude_columns or set()

    source_columns = set(source_df.columns)
    target_columns = set(spark.table(target_table).columns)

    common_columns = [
        c
        for c in source_df.columns
        if (
            c in source_columns
            and c in target_columns
            and c not in exclude_columns
        )
    ]

    return {
        c: f"s.`{c}`"
        for c in common_columns
    }


def validate_merge_key(
    source_df: DataFrame,
    target_table: str,
    merge_keys: list[str],
):
    """
    Fail-fast nếu merge key bị thiếu ở source hoặc target.
    """

    source_columns = set(source_df.columns)
    target_columns = set(spark.table(target_table).columns)

    missing_source = [
        c for c in merge_keys
        if c not in source_columns
    ]

    missing_target = [
        c for c in merge_keys
        if c not in target_columns
    ]

    if missing_source:
        raise ValueError(
            f"Missing merge key(s) in source for {target_table}: "
            f"{missing_source}"
        )

    if missing_target:
        raise ValueError(
            f"Missing merge key(s) in target for {target_table}: "
            f"{missing_target}"
        )


# =============================================================================
# SCHEMA — POI_MASTER
# =============================================================================

master_schema = StructType([
    StructField("poi_id",              StringType(),       True),
    StructField("lang",                StringType(),       True),
    StructField("slug",                StringType(),       True),
    StructField("poi_name",            StringType(),       True),
    StructField("business_sector",     StringType(),       True),
    StructField("business_category",   StringType(),       True),
    StructField("subcategory_tags",    StringType(),       True),
    StructField("operating_status",    StringType(),       True),
    StructField("is_active",           BooleanType(),      True),
    StructField("is_verified",         BooleanType(),      True),
    StructField("short_description",   StringType(),       True),
    StructField("short_address",       StringType(),       True),
    StructField("district",            StringType(),       True),
    StructField("rating_overall",      DecimalType(3, 1),  True),
    StructField("rating_count",        LongType(),         True),
    StructField("destination_ids",     StringType(),       True),
    StructField("nearby_pois",         StringType(),       True),
    StructField("destination_names",   StringType(),       True),
    StructField("created_at",          TimestampType(),    True),
    StructField("updated_at",          TimestampType(),    True),
    StructField("version",             IntegerType(),      True),
    StructField("source_id",           StringType(),       True),
    StructField("event_id",            StringType(),       True),
    StructField("ref_type",            StringType(),       True),
    StructField("ref_business_sector", StringType(),       True),
    StructField("ingestion_source",    StringType(),       True),
    StructField("partner_id",          StringType(),       True),
    StructField("is_bookable",         BooleanType(),      True),
    StructField("is_recommended",         BooleanType(),      True),

    # -------------------------------------------------------------------------
    # KHÔNG thêm:
    #
    # poi_group_id
    #
    # Vì Partner Portal không phải owner của 2 field này.
    # -------------------------------------------------------------------------
])


# =============================================================================
# SCHEMA — POI_DETAIL
# =============================================================================

detail_schema = StructType([
    StructField("poi_id",                 StringType(),    True),
    StructField("lang",                   StringType(),    True),
    StructField("amenities_json",         StringType(),    True),
    StructField("poi_address_json",       StringType(),    True),
    StructField("poi_contact_json",       StringType(),    True),
    StructField("poi_content_json",       StringType(),    True),
    StructField("poi_media_json",         StringType(),    True),
    StructField("poi_opening_hours_json", StringType(),    True),
    StructField("poi_policy_json",        StringType(),    True),
    StructField("poi_rating_json",        StringType(),    True),
    StructField("poi_review_json",        StringType(),    True),
    StructField("poi_extra_json",         StringType(),    True),
    StructField("source_id",              StringType(),    True),
    StructField("event_id",               StringType(),    True),
    StructField("ingestion_source",       StringType(),    True),
    StructField("created_at",             TimestampType(), True),
    StructField("updated_at",             TimestampType(), True),
])


# =============================================================================
# 8.1 — MERGE POI_MASTER
# =============================================================================

if master_rows:

    master_df = to_spark(
        master_rows,
        master_schema
    )

    print(
        f"✓ poi_master DataFrame: "
        f"{len(master_rows):,} rows"
    )

    # -------------------------------------------------------------------------
    # Validate merge key
    # -------------------------------------------------------------------------

    validate_merge_key(
        source_df=master_df,
        target_table=GLD_POI_MASTER,
        merge_keys=["poi_id"],
    )

    # -------------------------------------------------------------------------
    # Build INSERT mapping an toàn
    #
    # Vì master_df KHÔNG có:
    #
    #   poi_group_id
    #   is_recommended
    #
    # nên chúng sẽ không nằm trong insert map.
    #
    # Record mới:
    #   poi_group_id   = NULL
    #   is_recommended = NULL
    #
    # -------------------------------------------------------------------------

    master_insert_map = build_safe_insert_map(
        source_df=master_df,
        target_table=GLD_POI_MASTER,
    )

    print(
        "poi_master INSERT columns: "
        f"{len(master_insert_map)}"
    )

    (
        DeltaTable
        .forName(
            spark,
            GLD_POI_MASTER
        )
        .alias("t")

        .merge(
            master_df.alias("s"),
            "t.poi_id = s.poi_id"
        )

        # =====================================================================
        # UPDATE EXISTING POI
        # =====================================================================
        #
        # Không update:
        #
        #   poi_group_id
        #   is_recommended
        #
        # => nếu các field này đã được process khác set,
        #    giá trị sẽ được giữ nguyên.
        #
        # =====================================================================

        .whenMatchedUpdate(
            set={
                "lang":
                    "s.lang",

                "slug":
                    "s.slug",

                "poi_name":
                    "COALESCE(s.poi_name, '')",

                "business_sector":
                    "s.business_sector",

                "business_category":
                    "COALESCE("
                    "s.business_category, "
                    "t.business_category"
                    ")",

                "subcategory_tags":
                    "COALESCE("
                    "t.subcategory_tags, "
                    "s.subcategory_tags"
                    ")",

                "operating_status":
                    "s.operating_status",

                "is_active":
                    "s.is_active",

                "is_verified":
                    "s.is_verified",

                "short_description":
                    "s.short_description",

                "short_address":
                    "s.short_address",

                "district":
                    "s.district",

                # -------------------------------------------------------------
                # Giữ rating đã enrich từ nguồn khác nếu target đã có
                # -------------------------------------------------------------

                "rating_overall":
                    "COALESCE("
                    "t.rating_overall, "
                    "s.rating_overall"
                    ")",

                "rating_count":
                    """
                    CASE
                        WHEN t.rating_count > 0
                            THEN t.rating_count
                        ELSE s.rating_count
                    END
                    """,

                "destination_ids":
                    "s.destination_ids",

                "nearby_pois":
                    "COALESCE("
                    "t.nearby_pois, "
                    "s.nearby_pois"
                    ")",

                "destination_names":
                    "s.destination_names",

                # -------------------------------------------------------------
                # created_at KHÔNG update
                # -------------------------------------------------------------

                "updated_at":
                    "s.updated_at",

                "version":
                    "COALESCE(t.version, 0) + 1",

                "source_id":
                    "s.source_id",

                "ref_type":
                    "COALESCE("
                    "t.ref_type, "
                    "s.ref_type"
                    ")",

                "ref_business_sector":
                    "COALESCE("
                    "t.ref_business_sector, "
                    "s.ref_business_sector"
                    ")",

                "ingestion_source":
                    "s.ingestion_source",

                "partner_id":
                    "s.partner_id",

                "is_bookable":
                    "s.is_bookable",
                
                
                "is_recommended":
                    "s.is_recommended",

                # -------------------------------------------------------------
                # INTENTIONALLY NOT UPDATED
                # -------------------------------------------------------------
                #
                # "poi_group_id"
                # "is_recommended"
                #
            }
        )

        # =====================================================================
        # INSERT NEW POI
        # =====================================================================
        #
        # Chỉ insert các field mà source thực sự quản lý.
        #
        # Target fields không có trong master_df:
        #
        #   poi_group_id
        #   is_recommended
        #
        # sẽ tự nhận NULL/default.
        #
        # =====================================================================

        .whenNotMatchedInsert(
            values=master_insert_map
        )

        .execute()
    )

    print("✓ MERGE poi_master done")


# =============================================================================
# 8.2 — MERGE POI_DETAIL
# =============================================================================

if detail_rows:

    detail_df = to_spark(
        detail_rows,
        detail_schema
    )

    print(
        f"✓ poi_detail DataFrame: "
        f"{len(detail_rows):,} rows"
    )

    validate_merge_key(
        source_df=detail_df,
        target_table=GLD_POI_DETAIL,
        merge_keys=["poi_id"],
    )

    detail_insert_map = build_safe_insert_map(
        source_df=detail_df,
        target_table=GLD_POI_DETAIL,
    )

    (
        DeltaTable
        .forName(
            spark,
            GLD_POI_DETAIL
        )
        .alias("t")

        .merge(
            detail_df.alias("s"),
            "t.poi_id = s.poi_id"
        )

        .whenMatchedUpdate(
            set={
                "lang":
                    "s.lang",

                "amenities_json":
                    "s.amenities_json",

                "poi_address_json":
                    "s.poi_address_json",

                "poi_contact_json":
                    "s.poi_contact_json",

                "poi_content_json":
                    "s.poi_content_json",

                "poi_media_json":
                    "s.poi_media_json",

                "poi_opening_hours_json":
                    "s.poi_opening_hours_json",

                "poi_policy_json":
                    "s.poi_policy_json",

                "poi_rating_json":
                    "s.poi_rating_json",

                "poi_review_json":
                    "s.poi_review_json",

                "poi_extra_json":
                    "s.poi_extra_json",

                "source_id":
                    "s.source_id",

                "event_id":
                    "s.event_id",

                "ingestion_source":
                    "s.ingestion_source",

                # created_at không update

                "updated_at":
                    "s.updated_at",
            }
        )

        .whenNotMatchedInsert(
            values=detail_insert_map
        )

        .execute()
    )

    print("✓ MERGE poi_detail done")


# =============================================================================
# 8.3 — SCHEMA AI_POI
# =============================================================================
#
# Explicit schema:
# inferSchema có thể fail khi toàn bộ batch của một boolean/decimal field = None.
#
# =============================================================================

if ai_poi_rows:

    ai_poi_schema = StructType([

        StructField(
            "poi_id",
            StringType(),
            True
        ),

        StructField(
            "lang",
            StringType(),
            True
        ),

        StructField(
            "slug",
            StringType(),
            True
        ),

        StructField(
            "poi_name",
            StringType(),
            True
        ),

        StructField(
            "alternative_name",
            StringType(),
            True
        ),

        StructField(
            "business_sector",
            StringType(),
            True
        ),

        StructField(
            "business_category",
            StringType(),
            True
        ),

        StructField(
            "subcategory_tags",
            StringType(),
            True
        ),

        StructField(
            "ingestion_source",
            StringType(),
            True
        ),

        StructField(
            "poi_types",
            StringType(),
            True
        ),

        StructField(
            "destination_id",
            StringType(),
            True
        ),

        StructField(
            "destination_name",
            StringType(),
            True
        ),

        StructField(
            "is_active",
            BooleanType(),
            True
        ),

        StructField(
            "is_verified",
            BooleanType(),
            True
        ),

        StructField(
            "brand_or_chain",
            BooleanType(),
            True
        ),

        StructField(
            "short_address",
            StringType(),
            True
        ),

        StructField(
            "address_line_1",
            StringType(),
            True
        ),

        StructField(
            "address_line_2",
            StringType(),
            True
        ),

        StructField(
            "ward",
            StringType(),
            True
        ),

        StructField(
            "district",
            StringType(),
            True
        ),

        StructField(
            "province_name",
            StringType(),
            True
        ),

        StructField(
            "country_code",
            StringType(),
            True
        ),

        StructField(
            "lat",
            DoubleType(),
            True
        ),

        StructField(
            "lng",
            DoubleType(),
            True
        ),

        StructField(
            "phone",
            StringType(),
            True
        ),

        StructField(
            "website_url",
            StringType(),
            True
        ),

        StructField(
            "rating_overall",
            DecimalType(3, 1),
            True
        ),

        StructField(
            "rating_count",
            LongType(),
            True
        ),

        StructField(
            "price_level",
            IntegerType(),
            True
        ),

        StructField(
            "price_min",
            IntegerType(),
            True
        ),

        StructField(
            "price_max",
            IntegerType(),
            True
        ),

        StructField(
            "cover_thumbnail_url",
            StringType(),
            True
        ),

        StructField(
            "cover_original_url",
            StringType(),
            True
        ),

        StructField(
            "gallery_urls",
            StringType(),
            True
        ),

        StructField(
            "story",
            StringType(),
            True
        ),

        StructField(
            "description",
            StringType(),
            True
        ),

        StructField(
            "usp_highlights",
            StringType(),
            True
        ),

        StructField(
            "local_tips_or_neighborhood",
            StringType(),
            True
        ),

        StructField(
            "operating_hours",
            StringType(),
            True
        ),

        StructField(
            "is_24_7",
            BooleanType(),
            True
        ),

        StructField(
            "holiday_exceptions",
            StringType(),
            True
        ),

        StructField(
            "cancellation_policy",
            StringType(),
            True
        ),

        StructField(
            "takeout",
            BooleanType(),
            True
        ),

        StructField(
            "delivery",
            BooleanType(),
            True
        ),

        StructField(
            "dine_in",
            BooleanType(),
            True
        ),

        StructField(
            "curbside_pickup",
            BooleanType(),
            True
        ),

        StructField(
            "reservable",
            BooleanType(),
            True
        ),

        StructField(
            "serves_breakfast",
            BooleanType(),
            True
        ),

        StructField(
            "serves_lunch",
            BooleanType(),
            True
        ),

        StructField(
            "serves_dinner",
            BooleanType(),
            True
        ),

        StructField(
            "serves_beer",
            BooleanType(),
            True
        ),

        StructField(
            "serves_wine",
            BooleanType(),
            True
        ),

        StructField(
            "serves_brunch",
            BooleanType(),
            True
        ),

        StructField(
            "serves_vegetarian_food",
            BooleanType(),
            True
        ),

        StructField(
            "outdoor_seating",
            BooleanType(),
            True
        ),

        StructField(
            "live_music",
            BooleanType(),
            True
        ),

        StructField(
            "menu_for_children",
            BooleanType(),
            True
        ),

        StructField(
            "serves_cocktails",
            BooleanType(),
            True
        ),

        StructField(
            "serves_dessert",
            BooleanType(),
            True
        ),

        StructField(
            "serves_coffee",
            BooleanType(),
            True
        ),

        StructField(
            "good_for_children",
            BooleanType(),
            True
        ),

        StructField(
            "allows_dogs",
            BooleanType(),
            True
        ),

        StructField(
            "has_restroom",
            BooleanType(),
            True
        ),

        StructField(
            "good_for_groups",
            BooleanType(),
            True
        ),

        StructField(
            "good_for_watching_sports",
            BooleanType(),
            True
        ),

        StructField(
            "pure_service_area_business",
            BooleanType(),
            True
        ),

        StructField(
            "parking",
            BooleanType(),
            True
        ),

        StructField(
            "ev_charge",
            BooleanType(),
            True
        ),

        StructField(
            "payment_options_accepts_credit_cards",
            BooleanType(),
            True
        ),

        StructField(
            "payment_options_accepts_debit_cards",
            BooleanType(),
            True
        ),

        StructField(
            "payment_options_accepts_nfc",
            BooleanType(),
            True
        ),

        StructField(
            "parking_options_free_parking_lot",
            BooleanType(),
            True
        ),

        StructField(
            "parking_options_paid_parking_lot",
            BooleanType(),
            True
        ),

        StructField(
            "parking_options_free_street_parking",
            BooleanType(),
            True
        ),

        StructField(
            "parking_options_paid_street_parking",
            BooleanType(),
            True
        ),

        StructField(
            "parking_options_valet_parking",
            BooleanType(),
            True
        ),

        StructField(
            "parking_options_free_garage_parking",
            BooleanType(),
            True
        ),

        StructField(
            "parking_options_paid_garage_parking",
            BooleanType(),
            True
        ),

        StructField(
            "accessibility_options_wheelchair_accessible_parking",
            BooleanType(),
            True
        ),

        StructField(
            "accessibility_options_wheelchair_accessible_entrance",
            BooleanType(),
            True
        ),

        StructField(
            "accessibility_options_wheelchair_accessible_restroom",
            BooleanType(),
            True
        ),

        StructField(
            "accessibility_options_wheelchair_accessible_seating",
            BooleanType(),
            True
        ),

        StructField(
            "general_amenities",
            StringType(),
            True
        ),

        StructField(
            "created_at",
            TimestampType(),
            True
        ),

        StructField(
            "updated_at",
            TimestampType(),
            True
        ),

        StructField(
            "source_id",
            StringType(),
            True
        ),

        StructField(
            "event_id",
            StringType(),
            True
        ),

        StructField(
            "has_swimming_pool",
            BooleanType(),
            True
        ),

        StructField(
            "has_gym",
            BooleanType(),
            True
        ),

        StructField(
            "has_spa",
            BooleanType(),
            True
        ),

        StructField(
            "has_sauna",
            BooleanType(),
            True
        ),

        StructField(
            "has_garden",
            BooleanType(),
            True
        ),

        StructField(
            "has_bar_on_site",
            BooleanType(),
            True
        ),

        StructField(
            "has_restaurant_on_site",
            BooleanType(),
            True
        ),

        StructField(
            "has_coworking_space",
            BooleanType(),
            True
        ),

        StructField(
            "brand_name",
            StringType(),
            True
        ),

        StructField(
            "wifi_free",
            BooleanType(),
            True
        ),

        StructField(
            "has_jacuzzi",
            BooleanType(),
            True
        ),

        StructField(
            "has_meeting_rooms",
            BooleanType(),
            True
        ),

        StructField(
            "has_airport_shuttle",
            BooleanType(),
            True
        ),

        StructField(
            "front_desk_24h",
            BooleanType(),
            True
        ),

        StructField(
            "luggage_storage",
            BooleanType(),
            True
        ),

        StructField(
            "currency_exchange",
            BooleanType(),
            True
        ),

        StructField(
            "air_conditioning",
            BooleanType(),
            True
        ),

        StructField(
            "free_cancellation",
            BooleanType(),
            True
        ),

        StructField(
            "lowered_sink",
            BooleanType(),
            True
        ),

        StructField(
            "braille_elevator",
            BooleanType(),
            True
        ),
    ])


    # =========================================================================
    # BUILD AI_POI DATAFRAME
    # =========================================================================

    ai_poi_df = spark.createDataFrame(
        ai_poi_rows,
        schema=ai_poi_schema
    )

    print(
        f"✓ ai_poi DataFrame: "
        f"{len(ai_poi_rows):,} rows"
    )


    validate_merge_key(
        source_df=ai_poi_df,
        target_table=GLD_AI_POI,
        merge_keys=["poi_id"],
    )


    ai_poi_insert_map = build_safe_insert_map(
        source_df=ai_poi_df,
        target_table=GLD_AI_POI,
    )


    # =========================================================================
    # MERGE AI_POI
    # =========================================================================

    (
        DeltaTable
        .forName(
            spark,
            GLD_AI_POI
        )
        .alias("t")

        .merge(
            ai_poi_df.alias("s"),
            "t.poi_id = s.poi_id"
        )

        .whenMatchedUpdate(
            set={
                "lang":
                    "s.lang",

                "slug":
                    "s.slug",

                "poi_name":
                    "COALESCE(s.poi_name, '')",

                "business_sector":
                    "s.business_sector",

                "business_category":
                    "COALESCE("
                    "s.business_category, "
                    "t.business_category"
                    ")",

                "is_active":
                    "s.is_active",

                "is_verified":
                    "s.is_verified",

                "short_address":
                    "s.short_address",

                "address_line_1":
                    "s.address_line_1",

                "ward":
                    "s.ward",

                "district":
                    "s.district",

                "province_name":
                    "s.province_name",

                "lat":
                    "COALESCE(t.lat, s.lat)",

                "lng":
                    "COALESCE(t.lng, s.lng)",

                "phone":
                    "s.phone",

                "website_url":
                    "s.website_url",

                "cover_thumbnail_url":
                    "COALESCE("
                    "s.cover_thumbnail_url, "
                    "t.cover_thumbnail_url"
                    ")",

                "cover_original_url":
                    "COALESCE("
                    "s.cover_original_url, "
                    "t.cover_original_url"
                    ")",

                "gallery_urls":
                    "COALESCE("
                    "s.gallery_urls, "
                    "t.gallery_urls"
                    ")",

                "story":
                    "s.story",

                "description":
                    "s.description",

                "usp_highlights":
                    "s.usp_highlights",

                "local_tips_or_neighborhood":
                    "s.local_tips_or_neighborhood",

                "operating_hours":
                    "COALESCE("
                    "s.operating_hours, "
                    "t.operating_hours"
                    ")",

                "is_24_7":
                    "s.is_24_7",

                "source_id":
                    "s.source_id",

                "updated_at":
                    "s.updated_at",

                # -------------------------------------------------------------
                # Amenity flags
                # -------------------------------------------------------------

                "takeout":
                    "s.takeout",

                "delivery":
                    "s.delivery",

                "dine_in":
                    "s.dine_in",

                "reservable":
                    "s.reservable",

                "serves_breakfast":
                    "s.serves_breakfast",

                "serves_lunch":
                    "s.serves_lunch",

                "serves_dinner":
                    "s.serves_dinner",

                "serves_beer":
                    "s.serves_beer",

                "serves_wine":
                    "s.serves_wine",

                "serves_brunch":
                    "s.serves_brunch",

                "serves_vegetarian_food":
                    "s.serves_vegetarian_food",

                "outdoor_seating":
                    "s.outdoor_seating",

                "live_music":
                    "s.live_music",

                "menu_for_children":
                    "s.menu_for_children",

                "serves_cocktails":
                    "s.serves_cocktails",

                "serves_dessert":
                    "s.serves_dessert",

                "serves_coffee":
                    "s.serves_coffee",

                "good_for_children":
                    "s.good_for_children",

                "allows_dogs":
                    "s.allows_dogs",

                "has_restroom":
                    "s.has_restroom",

                "good_for_groups":
                    "s.good_for_groups",

                "good_for_watching_sports":
                    "s.good_for_watching_sports",

                "pure_service_area_business":
                    "s.pure_service_area_business",

                "parking":
                    "s.parking",

                "ev_charge":
                    "s.ev_charge",

                "payment_options_accepts_credit_cards":
                    "s.payment_options_accepts_credit_cards",

                "payment_options_accepts_debit_cards":
                    "s.payment_options_accepts_debit_cards",

                "payment_options_accepts_nfc":
                    "s.payment_options_accepts_nfc",

                "accessibility_options_wheelchair_accessible_parking":
                    "s.accessibility_options_wheelchair_accessible_parking",

                "accessibility_options_wheelchair_accessible_entrance":
                    "s.accessibility_options_wheelchair_accessible_entrance",

                "accessibility_options_wheelchair_accessible_restroom":
                    "s.accessibility_options_wheelchair_accessible_restroom",

                "general_amenities":
                    "s.general_amenities",

                "has_swimming_pool":
                    "s.has_swimming_pool",

                "has_gym":
                    "s.has_gym",

                "has_spa":
                    "s.has_spa",

                "has_sauna":
                    "s.has_sauna",

                "has_garden":
                    "s.has_garden",

                "has_bar_on_site":
                    "s.has_bar_on_site",

                "has_restaurant_on_site":
                    "s.has_restaurant_on_site",

                "has_coworking_space":
                    "s.has_coworking_space",

                "wifi_free":
                    "s.wifi_free",

                "has_jacuzzi":
                    "s.has_jacuzzi",

                "has_meeting_rooms":
                    "s.has_meeting_rooms",

                "has_airport_shuttle":
                    "s.has_airport_shuttle",

                "front_desk_24h":
                    "s.front_desk_24h",

                "luggage_storage":
                    "s.luggage_storage",

                "currency_exchange":
                    "s.currency_exchange",

                "air_conditioning":
                    "s.air_conditioning",

                "free_cancellation":
                    "s.free_cancellation",

                "lowered_sink":
                    "s.lowered_sink",

                "braille_elevator":
                    "s.braille_elevator",
            }
        )

        .whenNotMatchedInsert(
            values=ai_poi_insert_map
        )

        .execute()
    )

    print("✓ MERGE ai_poi done")


print("✅ CELL 8 — all Gold MERGEs completed")

# def to_spark(rows: list, schema: StructType) -> DataFrame:
#     """Tạo DataFrame từ list of tuples với schema rõ ràng."""
#     return spark.createDataFrame(rows, schema=schema)


# # --- Schema poi_master ---
# master_schema = StructType([
#     StructField("poi_id",              StringType(),       True),
#     StructField("lang",                StringType(),       True),
#     StructField("slug",                StringType(),       True),
#     StructField("poi_name",            StringType(),       True),
#     StructField("business_sector",     StringType(),       True),
#     StructField("business_category",   StringType(),       True),
#     StructField("subcategory_tags",    StringType(),       True),
#     StructField("operating_status",    StringType(),       True),
#     StructField("is_active",           BooleanType(),      True),
#     StructField("is_verified",         BooleanType(),      True),
#     StructField("short_description",   StringType(),       True),
#     StructField("short_address",       StringType(),       True),
#     StructField("district",            StringType(),       True),
#     StructField("rating_overall",      DecimalType(3,1),   True),
#     StructField("rating_count",        LongType(),         True),
#     StructField("destination_ids",     StringType(),       True),
#     StructField("nearby_pois",         StringType(),       True),
#     StructField("destination_names",   StringType(),       True),
#     StructField("created_at",          TimestampType(),    True),
#     StructField("updated_at",          TimestampType(),    True),
#     StructField("version",             IntegerType(),      True),
#     StructField("source_id",           StringType(),       True),
#     StructField("event_id",            StringType(),       True),
#     StructField("ref_type",            StringType(),       True),
#     StructField("ref_business_sector", StringType(),       True),
#     StructField("ingestion_source",    StringType(),       True),
#     StructField("partner_id",          StringType(),       True),
#     StructField("is_bookable",         BooleanType(),      True),
# ])

# # --- Schema poi_detail ---
# detail_schema = StructType([
#     StructField("poi_id",                StringType(),   True),
#     StructField("lang",                  StringType(),   True),
#     StructField("amenities_json",        StringType(),   True),
#     StructField("poi_address_json",      StringType(),   True),
#     StructField("poi_contact_json",      StringType(),   True),
#     StructField("poi_content_json",      StringType(),   True),
#     StructField("poi_media_json",        StringType(),   True),
#     StructField("poi_opening_hours_json",StringType(),   True),
#     StructField("poi_policy_json",       StringType(),   True),
#     StructField("poi_rating_json",       StringType(),   True),
#     StructField("poi_review_json",       StringType(),   True),
#     StructField("poi_extra_json",        StringType(),   True),
#     StructField("source_id",             StringType(),   True),
#     StructField("event_id",              StringType(),   True),
#     StructField("ingestion_source",      StringType(),   True),
#     StructField("created_at",            TimestampType(),True),
#     StructField("updated_at",            TimestampType(),True),
# ])

# # ai_poi có quá nhiều cột boolean → dùng inferSchema từ dữ liệu mẫu
# # nhưng vẫn đảm bảo None-safe bằng cách ép kiểu trước
# if master_rows:
#     master_df = to_spark(master_rows, master_schema)
#     print(f"✓ poi_master DataFrame: {len(master_rows):,} rows")

#     (
#         DeltaTable.forName(spark, GLD_POI_MASTER).alias("t")
#         .merge(master_df.alias("s"), "t.poi_id = s.poi_id")
#         .whenMatchedUpdate(set={
#             "lang":                 "s.lang",
#             "slug":                 "s.slug",
#             "poi_name":             "COALESCE(s.poi_name, '')",
#             "business_sector":      "s.business_sector",
#             "business_category":    "COALESCE(s.business_category, t.business_category)",
#             "subcategory_tags":     "COALESCE(t.subcategory_tags, s.subcategory_tags)",
#             "operating_status":     "s.operating_status",
#             "is_active":            "s.is_active",
#             "is_verified":          "s.is_verified",
#             "short_description":    "s.short_description",
#             "short_address":        "s.short_address",
#             "district":             "s.district",
#             "rating_overall":       "COALESCE(t.rating_overall, s.rating_overall)",
#             "rating_count":         "CASE WHEN t.rating_count > 0 THEN t.rating_count ELSE s.rating_count END",
#             "destination_ids":      "s.destination_ids",
#             "nearby_pois":          "COALESCE(t.nearby_pois, s.nearby_pois)",
#             "destination_names":    "s.destination_names",
#             "updated_at":           "s.updated_at",
#             "version":              "t.version + 1",
#             "source_id":            "s.source_id",
#             "ref_type":             "COALESCE(t.ref_type, s.ref_type)",
#             "ref_business_sector":  "COALESCE(t.ref_business_sector, s.ref_business_sector)",
#             "ingestion_source":     "s.ingestion_source",
#             "partner_id":           "s.partner_id",
#             "is_bookable":          "s.is_bookable",
#         })
#         .whenNotMatchedInsertAll()
#         .execute()
#     )
#     print(f"✓ MERGE poi_master done")

# if detail_rows:
#     detail_df = to_spark(detail_rows, detail_schema)
#     print(f"✓ poi_detail DataFrame: {len(detail_rows):,} rows")

#     (
#         DeltaTable.forName(spark, GLD_POI_DETAIL).alias("t")
#         .merge(detail_df.alias("s"), "t.poi_id = s.poi_id")
#         .whenMatchedUpdate(set={
#             "lang":                    "s.lang",
#             "amenities_json":          "s.amenities_json",
#             "poi_address_json":        "s.poi_address_json",
#             "poi_contact_json":        "s.poi_contact_json",
#             "poi_content_json":        "s.poi_content_json",
#             "poi_media_json":          "s.poi_media_json",
#             "poi_opening_hours_json":  "s.poi_opening_hours_json",
#             "poi_policy_json":         "s.poi_policy_json",
#             "poi_rating_json":         "s.poi_rating_json",
#             "poi_review_json":         "s.poi_review_json",
#             "poi_extra_json":          "s.poi_extra_json",
#             "source_id":               "s.source_id",
#             "event_id":                "s.event_id",
#             "ingestion_source":        "s.ingestion_source",
#             "updated_at":              "s.updated_at",
#         })
#         .whenNotMatchedInsertAll()
#         .execute()
#     )
#     print(f"✓ MERGE poi_detail done")

# # ai_poi: dùng explicit schema — infer schema fail khi có None trong boolean/decimal cols
# if ai_poi_rows:
#     ai_poi_schema = StructType([
#         StructField("poi_id",                                               StringType(),    True),
#         StructField("lang",                                                 StringType(),    True),
#         StructField("slug",                                                 StringType(),    True),
#         StructField("poi_name",                                             StringType(),    True),
#         StructField("alternative_name",                                     StringType(),    True),
#         StructField("business_sector",                                      StringType(),    True),
#         StructField("business_category",                                    StringType(),    True),
#         StructField("subcategory_tags",                                     StringType(),    True),
#         StructField("ingestion_source",                                     StringType(),    True),
#         StructField("poi_types",                                            StringType(),    True),
#         StructField("destination_id",                                       StringType(),    True),
#         StructField("destination_name",                                     StringType(),    True),
#         StructField("is_active",                                            BooleanType(),   True),
#         StructField("is_verified",                                          BooleanType(),   True),
#         StructField("brand_or_chain",                                       BooleanType(),   True),
#         StructField("short_address",                                        StringType(),    True),
#         StructField("address_line_1",                                       StringType(),    True),
#         StructField("address_line_2",                                       StringType(),    True),
#         StructField("ward",                                                 StringType(),    True),
#         StructField("district",                                             StringType(),    True),
#         StructField("province_name",                                        StringType(),    True),
#         StructField("country_code",                                         StringType(),    True),
#         StructField("lat",                                                  DoubleType(),    True),
#         StructField("lng",                                                  DoubleType(),    True),
#         StructField("phone",                                                StringType(),    True),
#         StructField("website_url",                                          StringType(),    True),
#         StructField("rating_overall",                                       DecimalType(3,1),True),
#         StructField("rating_count",                                         LongType(),      True),
#         StructField("price_level",                                          IntegerType(),   True),
#         StructField("price_min",                                            IntegerType(),   True),
#         StructField("price_max",                                            IntegerType(),   True),
#         StructField("cover_thumbnail_url",                                  StringType(),    True),
#         StructField("cover_original_url",                                   StringType(),    True),
#         StructField("gallery_urls",                                         StringType(),    True),
#         StructField("story",                                                StringType(),    True),
#         StructField("description",                                          StringType(),    True),
#         StructField("usp_highlights",                                       StringType(),    True),
#         StructField("local_tips_or_neighborhood",                           StringType(),    True),
#         StructField("operating_hours",                                      StringType(),    True),
#         StructField("is_24_7",                                              BooleanType(),   True),
#         StructField("holiday_exceptions",                                   StringType(),    True),
#         StructField("cancellation_policy",                                  StringType(),    True),
#         StructField("takeout",                                              BooleanType(),   True),
#         StructField("delivery",                                             BooleanType(),   True),
#         StructField("dine_in",                                              BooleanType(),   True),
#         StructField("curbside_pickup",                                      BooleanType(),   True),
#         StructField("reservable",                                           BooleanType(),   True),
#         StructField("serves_breakfast",                                     BooleanType(),   True),
#         StructField("serves_lunch",                                         BooleanType(),   True),
#         StructField("serves_dinner",                                        BooleanType(),   True),
#         StructField("serves_beer",                                          BooleanType(),   True),
#         StructField("serves_wine",                                          BooleanType(),   True),
#         StructField("serves_brunch",                                        BooleanType(),   True),
#         StructField("serves_vegetarian_food",                               BooleanType(),   True),
#         StructField("outdoor_seating",                                      BooleanType(),   True),
#         StructField("live_music",                                           BooleanType(),   True),
#         StructField("menu_for_children",                                    BooleanType(),   True),
#         StructField("serves_cocktails",                                     BooleanType(),   True),
#         StructField("serves_dessert",                                       BooleanType(),   True),
#         StructField("serves_coffee",                                        BooleanType(),   True),
#         StructField("good_for_children",                                    BooleanType(),   True),
#         StructField("allows_dogs",                                          BooleanType(),   True),
#         StructField("has_restroom",                                         BooleanType(),   True),
#         StructField("good_for_groups",                                      BooleanType(),   True),
#         StructField("good_for_watching_sports",                             BooleanType(),   True),
#         StructField("pure_service_area_business",                           BooleanType(),   True),
#         StructField("parking",                                              BooleanType(),   True),
#         StructField("ev_charge",                                            BooleanType(),   True),
#         StructField("payment_options_accepts_credit_cards",                 BooleanType(),   True),
#         StructField("payment_options_accepts_debit_cards",                  BooleanType(),   True),
#         StructField("payment_options_accepts_nfc",                          BooleanType(),   True),
#         StructField("parking_options_free_parking_lot",                     BooleanType(),   True),
#         StructField("parking_options_paid_parking_lot",                     BooleanType(),   True),
#         StructField("parking_options_free_street_parking",                  BooleanType(),   True),
#         StructField("parking_options_paid_street_parking",                  BooleanType(),   True),
#         StructField("parking_options_valet_parking",                        BooleanType(),   True),
#         StructField("parking_options_free_garage_parking",                  BooleanType(),   True),
#         StructField("parking_options_paid_garage_parking",                  BooleanType(),   True),
#         StructField("accessibility_options_wheelchair_accessible_parking",  BooleanType(),   True),
#         StructField("accessibility_options_wheelchair_accessible_entrance", BooleanType(),   True),
#         StructField("accessibility_options_wheelchair_accessible_restroom", BooleanType(),   True),
#         StructField("accessibility_options_wheelchair_accessible_seating",  BooleanType(),   True),
#         StructField("general_amenities",                                    StringType(),    True),
#         StructField("created_at",                                           TimestampType(), True),
#         StructField("updated_at",                                           TimestampType(), True),
#         StructField("source_id",                                            StringType(),    True),
#         StructField("event_id",                                             StringType(),    True),
#         StructField("has_swimming_pool",                                    BooleanType(),   True),
#         StructField("has_gym",                                              BooleanType(),   True),
#         StructField("has_spa",                                              BooleanType(),   True),
#         StructField("has_sauna",                                            BooleanType(),   True),
#         StructField("has_garden",                                           BooleanType(),   True),
#         StructField("has_bar_on_site",                                      BooleanType(),   True),
#         StructField("has_restaurant_on_site",                               BooleanType(),   True),
#         StructField("has_coworking_space",                                  BooleanType(),   True),
#         StructField("brand_name",                                           StringType(),    True),
#         StructField("wifi_free",                                            BooleanType(),   True),
#         StructField("has_jacuzzi",                                          BooleanType(),   True),
#         StructField("has_meeting_rooms",                                    BooleanType(),   True),
#         StructField("has_airport_shuttle",                                  BooleanType(),   True),
#         StructField("front_desk_24h",                                       BooleanType(),   True),
#         StructField("luggage_storage",                                      BooleanType(),   True),
#         StructField("currency_exchange",                                    BooleanType(),   True),
#         StructField("air_conditioning",                                     BooleanType(),   True),
#         StructField("free_cancellation",                                    BooleanType(),   True),
#         StructField("lowered_sink",                                         BooleanType(),   True),
#         StructField("braille_elevator",                                     BooleanType(),   True),
#     ])
#     ai_poi_df = spark.createDataFrame(ai_poi_rows, schema=ai_poi_schema)
#     print(f"✓ ai_poi DataFrame: {ai_poi_df.count():,} rows")

#     (
#         DeltaTable.forName(spark, GLD_AI_POI).alias("t")
#         .merge(ai_poi_df.alias("s"), "t.poi_id = s.poi_id")
#         .whenMatchedUpdate(set={
#             "lang":             "s.lang",
#             "slug":             "s.slug",
#             "poi_name":         "COALESCE(s.poi_name, '')",
#             "business_sector":  "s.business_sector",
#             "business_category": "COALESCE(s.business_category, t.business_category)",
#             "is_active":        "s.is_active",
#             "is_verified":      "s.is_verified",
#             "short_address":    "s.short_address",
#             "address_line_1":   "s.address_line_1",
#             "ward":             "s.ward",
#             "district":         "s.district",
#             "province_name":    "s.province_name",
#             "lat":              "COALESCE(t.lat, s.lat)",
#             "lng":              "COALESCE(t.lng, s.lng)",
#             "phone":            "s.phone",
#             "website_url":      "s.website_url",
#             "cover_thumbnail_url": "COALESCE(s.cover_thumbnail_url, t.cover_thumbnail_url)",
#             "cover_original_url":  "COALESCE(s.cover_original_url, t.cover_original_url)",
#             "gallery_urls":     "COALESCE(s.gallery_urls, t.gallery_urls)",
#             "story":            "s.story",
#             "description":      "s.description",
#             "usp_highlights":             "s.usp_highlights",
#             "local_tips_or_neighborhood": "s.local_tips_or_neighborhood",
#             "operating_hours":  "COALESCE(s.operating_hours, t.operating_hours)",
#             "is_24_7":          "s.is_24_7",
#             "source_id":        "s.source_id",
#             "updated_at":       "s.updated_at",
#             # Amenity flags (Confluence v1.5, mục 5.2) — refresh lại mỗi lần MERGE
#             "takeout":                                              "s.takeout",
#             "delivery":                                             "s.delivery",
#             "dine_in":                                              "s.dine_in",
#             "reservable":                                           "s.reservable",
#             "serves_breakfast":                                     "s.serves_breakfast",
#             "serves_lunch":                                         "s.serves_lunch",
#             "serves_dinner":                                        "s.serves_dinner",
#             "serves_beer":                                          "s.serves_beer",
#             "serves_wine":                                          "s.serves_wine",
#             "serves_brunch":                                        "s.serves_brunch",
#             "serves_vegetarian_food":                                "s.serves_vegetarian_food",
#             "outdoor_seating":                                      "s.outdoor_seating",
#             "live_music":                                           "s.live_music",
#             "menu_for_children":                                    "s.menu_for_children",
#             "serves_cocktails":                                     "s.serves_cocktails",
#             "serves_dessert":                                       "s.serves_dessert",
#             "serves_coffee":                                        "s.serves_coffee",
#             "good_for_children":                                    "s.good_for_children",
#             "allows_dogs":                                          "s.allows_dogs",
#             "has_restroom":                                         "s.has_restroom",
#             "good_for_groups":                                      "s.good_for_groups",
#             "good_for_watching_sports":                             "s.good_for_watching_sports",
#             "pure_service_area_business":                           "s.pure_service_area_business",
#             "parking":                                              "s.parking",
#             "ev_charge":                                            "s.ev_charge",
#             "payment_options_accepts_credit_cards":                 "s.payment_options_accepts_credit_cards",
#             "payment_options_accepts_debit_cards":                  "s.payment_options_accepts_debit_cards",
#             "payment_options_accepts_nfc":                          "s.payment_options_accepts_nfc",
#             "accessibility_options_wheelchair_accessible_parking":  "s.accessibility_options_wheelchair_accessible_parking",
#             "accessibility_options_wheelchair_accessible_entrance": "s.accessibility_options_wheelchair_accessible_entrance",
#             "accessibility_options_wheelchair_accessible_restroom": "s.accessibility_options_wheelchair_accessible_restroom",
#             "general_amenities":                                    "s.general_amenities",
#             "has_swimming_pool":                                    "s.has_swimming_pool",
#             "has_gym":                                              "s.has_gym",
#             "has_spa":                                              "s.has_spa",
#             "has_sauna":                                            "s.has_sauna",
#             "has_garden":                                           "s.has_garden",
#             "has_bar_on_site":                                      "s.has_bar_on_site",
#             "has_restaurant_on_site":                               "s.has_restaurant_on_site",
#             "has_coworking_space":                                  "s.has_coworking_space",
#             "wifi_free":                                            "s.wifi_free",
#             "has_jacuzzi":                                          "s.has_jacuzzi",
#             "has_meeting_rooms":                                    "s.has_meeting_rooms",
#             "has_airport_shuttle":                                  "s.has_airport_shuttle",
#             "front_desk_24h":                                       "s.front_desk_24h",
#             "luggage_storage":                                      "s.luggage_storage",
#             "currency_exchange":                                    "s.currency_exchange",
#             "air_conditioning":                                     "s.air_conditioning",
#             "free_cancellation":                                    "s.free_cancellation",
#             "lowered_sink":                                         "s.lowered_sink",
#             "braille_elevator":                                     "s.braille_elevator",
#         })
#         .whenNotMatchedInsertAll()
#         .execute()
#     )
#     print(f"✓ MERGE ai_poi done")


# %%
# =============================================================================
# CELL 9 — UPDATE WATERMARK
# =============================================================================

spark.createDataFrame([(
    WM_FLOW_ID,
    upper_s,
    RUN_ID,
    datetime.utcnow(),
    "SUCCESS",
)], schema=StructType([
    StructField("flow_id",         StringType(),   True),
    StructField("last_updated_at", LongType(),     True),
    StructField("last_run_id",     StringType(),   True),
    StructField("last_run_at",     TimestampType(),True),
    StructField("status",          StringType(),   True),
])).write.format("delta").mode("append").saveAsTable(WM_TABLE)

print(f"✓ Watermark advanced → {upper_s}")
print(f"  Tương đương: {datetime.utcfromtimestamp(upper_s)}")
print(f"\n✅ RUN SUCCESS — RUN_ID: {RUN_ID}")
print(f"   poi_master : {len(master_rows):,} rows")
print(f"   ai_poi     : {len(ai_poi_rows):,} rows")
print(f"   poi_detail : {len(detail_rows):,} rows")

final_df.unpersist()


# %%
# =============================================================================
# CELL 10 — TEST DATA FACTORY + UNIT TESTS
# =============================================================================
# Giải thích:
#   make_service() tạo 1 record mẫu với giá trị mặc định hợp lệ.
#   Override bất kỳ field nào bằng **kwargs.
#   Dùng trong test để kiểm tra từng hàm riêng biệt — không cần chạy
#   toàn bộ pipeline.
#
#   Ví dụ:
#     svc = make_service(sector="f_n_b", story=SENTINEL)
#     am  = parse_json(svc["amentities"])
#     pol = extract_policy(am, "f_n_b")
#     assert pol["pet_policy"] is not None
# =============================================================================

def make_amentities(sector: str = "accommodation", overrides: dict = None) -> dict:
    """
    Tạo amentities JSON mặc định cho 1 sector — đủ field để test mapping
    amenity (Confluence v1.5, mục 5.2): facilities.accessibility, menu[],
    store_profile, organizer_profile, logistics, fleet_profile...
    overrides: dict để ghi đè bất kỳ field nào (merge nông ở cấp top-level).
    """
    base = {
        "facilities": {
            "general_amenities": ["Free Wi-Fi", "Swimming Pool", "24-hour Front Desk"],
            "accessibility":     ["Accessible Parking", "Wheelchair Path", "Grab Rail Toilet",
                                   "Braille Elevator", "Lowered Sink"],
            "safety_and_hygiene": [],
        },
        "policies": {
            "operating_hours": {
                "monday":    {"status": True,  "hours": [{"open_time": "08:00", "close_time": "22:00"}]},
                "tuesday":   {"status": True,  "hours": [{"open_time": "08:00", "close_time": "22:00"}]},
                "wednesday": {"status": True,  "hours": [{"open_time": "08:00", "close_time": "22:00"}]},
                "thursday":  {"status": True,  "hours": [{"open_time": "08:00", "close_time": "22:00"}]},
                "friday":    {"status": True,  "hours": [{"open_time": "08:00", "close_time": "22:00"}]},
                "saturday":  {"status": True,  "hours": [{"open_time": "08:00", "close_time": "22:00"}]},
                "sunday":    {"status": False, "hours": []},
            },
            "check_in_out":    {"check_in_time": "14:00", "check_out_time": "12:00"},
            "breakfast_policy": True,
            "child_policy":    {"status": True},
            "pet_policy":      True,
            "cancellation_policy": {"status": True, "policy": "Standard Flexible"},
        },
    }

    if sector == "f_n_b":
        base["facilities"]["main_cuisine"]   = "Vietnamese"
        base["facilities"]["specialty_tags"] = ["Coffee Shop", "Office Lunch"]
        base["facilities"]["space_and_services"] = ["Live Music", "Outdoor Seating", "Table Service", "Sports Screening"]
        base["policies"]["house_rules"] = {"pet_allowed": True, "smoking": False}
        base["policies"]["deposit_and_cancellation"] = {"status": True, "cancellation_window_hours": 24}
        base["menu"] = [{
            "operational_tags":    ["Takeaway/Delivery valid"],
            "dietary_and_allergy": ["Alcohol/18+", "Vegetarian", "Kid-Friendly"],
            "commercial_category": ["Combo/Platter"],
        }]

    elif sector == "attraction":
        base["facilities"]["classification"] = {
            "main_category":  "Theme Park",
            "specialty_tags": ["Adventure", "Family"],
        }
        base["facilities"]["facilities_and_services"] = ["Restroom", "Parking", "Free Wi-Fi", "Restaurant-Cafe", "Lockers"]
        base["policies"]["gate_and_entry_rules"] = {"age_restriction": "None"}
        base["policies"]["refund_and_change_policy"] = {"status": True, "cancellation_allowed": True}

    elif sector == "spa_n_wellness":
        base["center_profile"] = {
            "business_category": "Day Spa",
            "facilities_and_ambience": ["Relaxation room", "Steam room", "Air-conditioned",
                                         "Car Parking Available", "Outdoor/Al Fresco Seating", "Live Music"],
        }
        base["policies"]["health_restrictions"] = {"min_age": 0, "smoking": "prohibited"}
        base["policies"]["financial_transparency"] = {
            "native_booking": True,
            "tipping_policy": "optional",
        }
        base["policies"]["late_arrival_and_cancellation"] = {"free_cancellation_hours": 4}

    elif sector == "shopping_n_retail":
        base["store_profile"] = {
            "main_category": "Souvenir Shop",
            "facilities_and_services": ["Free Wi-Fi", "Parking", "Credit Card (Visa/Master)"],
        }
        base["policies"]["fulfillment_and_payment"] = {
            "supported_payments": ["Napas", "Apple Pay"],
            "shipping_coverage":  ["Local/City Delivery"],
        }
        base["policies"]["return_and_exchange"] = {"status": True}

    elif sector == "event":
        base["organizer_profile"] = {
            "core_competency":  "Concert Organizer",
            "specialty_topics": ["Concert-Live Music"],
        }
        base["facilities"]["event_services"] = ["Stage", "Sound System"]
        base["policies"]["venue_regulations"] = {"dress_code": "None"}
        base["policies"]["refund_and_postponement"] = {"status": True, "refund_allowed": True}

    elif sector == "tour_activities":
        base["logistics"] = {"fleet_amenities": ["On-board Wi-Fi", "Air-Conditioned"]}
        base["policies"]["cancellation_policy"] = {
            "rules": [{"penalty_percentage": 0}, {"penalty_percentage": 50}],
        }

    elif sector == "transport_route_base":
        base["fleet_profile"] = {
            "vehicle_types":      ["Sleeper Bus"],
            "on_board_amenities": ["Free Wi-Fi", "Air Conditioning", "On-board Toilet"],
        }
        base["policies"]["luggage_and_child_policy"] = {"free_infant_height_cm": 90}
        base["policies"]["cancellation_policy"] = {"cancellation_allowed": True}

    elif sector == "transport_on_demand":
        base["operator_profile"] = {"business_model": "E-hailing"}
        base["policies"]["cancellation_policy"] = {"free_cancellation_hours": 1}

    elif sector == "vehicle_rental":
        base["logistics"] = {"business_type": "Self-drive"}
        base["policies"]["cancellation_policy"] = {"free_cancellation_hours": 2}

    if overrides:
        base.update(overrides)

    return base


def make_service(
    sector:    str = "accommodation",
    name:      str = "Test Hotel",
    is_active: bool = True,
    **overrides
) -> dict:
    """
    Tạo 1 record business_service mẫu hợp lệ.
    Dùng làm fixture cho unit tests.

    Ví dụ:
        svc = make_service(sector="f_n_b", story=SENTINEL)
        svc = make_service(sector="attraction", is_active=False)
        svc = make_service(amentities=json.dumps({"custom": "data"}))
    """
    svc_id = str(uuid.uuid4())
    am_dict = make_amentities(sector)

    base = {
        "id":                   svc_id,
        "partner_id":           str(uuid.uuid4()),
        "svc_partner_id":        str(uuid.uuid4()),  # [F5] field Spark tạo trước join để tránh ambiguous
        "business_location_id": str(uuid.uuid4()),
        "name":                 name,
        "phone":                "0905123456",
        "hotline":              "19001234",
        "email":                f"test+{svc_id[:8]}@example.com",
        "business_sector":      sector,
        "operation_status":     "operational",
        "address":              json.dumps({
            "full_address": f"123 Nguyễn Huệ, Quận 1, TP.HCM",
            "street":       "Nguyễn Huệ",
            "ward":         "Phường Bến Nghé",
            "district":     "Quận 1",
            "city":         "TP.HCM",
        }),
        "bio":                  f"Mô tả ngắn về {name}",
        "story":                f"Câu chuyện thương hiệu {name}",
        "amentities":           json.dumps(am_dict),
        "media":                json.dumps([
            {"url": "https://example.com/cover.jpg", "order": 1, "is_cover": True},
            {"url": "https://example.com/img2.jpg",  "order": 2, "is_cover": False},
        ]),
        "is_main":              True,
        "is_active":            is_active,
        "last_updated_at":      int(datetime.utcnow().timestamp()),   # epoch seconds
        "created_at":           int(datetime.utcnow().timestamp()),    # epoch seconds
        "last_verify_at":       None,
        "last_in_activated_at": None,
        "website_urls":         json.dumps({"website_url": "https://example.com"}),
        # Joined fields (thường có sau join)
        "poi_id":               md5(f"{INGESTION}{svc_id}vi".encode()).hexdigest(),
        "partner_type":         "HOTEL",
        "company_legal_name":   f"{name} Co., Ltd.",
        "company_legal_vietnamese_name": f"Công ty {name}",
        "company_short_name":   name,
        "p_updated_at":         int(datetime.utcnow().timestamp()),    # epoch seconds
        "bl_updated_at":        int(datetime.utcnow().timestamp()),    # epoch seconds
        "addr_full":            "123 Nguyễn Huệ, Quận 1, TP.HCM",
        "addr_street":          "Nguyễn Huệ",
        "addr_ward":            "Phường Bến Nghé",
        "addr_district":        "Quận 1",
        "addr_city":            "TP.HCM",
        "lat":                  10.7769,
        "lng":                  106.7009,
        "dest_name":            "TP.HCM",
        "parent_dest_name":     "Việt Nam",
        "dest_id":              "hcm",
        "enriched_ward":        None,
        "enriched_district":    None,
        "enriched_province":    None,
        "is_bookable":          False,
    }
    base.update(overrides)
    return base


# =============================================================================
# UNIT TESTS
# =============================================================================

def run_tests():
    passed = 0
    failed = 0

    def ok(name):
        nonlocal passed
        passed += 1
        print(f"  ✓ {name}")

    def fail(name, msg):
        nonlocal failed
        failed += 1
        print(f"  ❌ {name}: {msg}")

    print("\n=== UNIT TESTS ===\n")

    # -------------------------------------------------------
    # Test 1: make_poi_id deterministic
    # -------------------------------------------------------
    id1 = make_poi_id("abc-123")
    id2 = make_poi_id("abc-123")
    if id1 == id2:
        ok("make_poi_id deterministic")
    else:
        fail("make_poi_id deterministic", f"{id1} != {id2}")

    # -------------------------------------------------------
    # Test 2: clean() bỏ sentinel
    # -------------------------------------------------------
    if clean(SENTINEL) is None:
        ok("clean() bỏ sentinel → None")
    else:
        fail("clean() sentinel", "phải trả về None")

    if clean("") is None:
        ok("clean() bỏ chuỗi rỗng → None")
    else:
        fail("clean() empty string", "phải trả về None")

    # -------------------------------------------------------
    # Test 3: [F3] short_address dùng full_address
    # -------------------------------------------------------
    svc = make_service(sector="accommodation")
    am  = parse_json(svc["amentities"])
    b   = build_row_base(svc, am)
    if b["addr_full"] == "123 Nguyễn Huệ, Quận 1, TP.HCM":
        ok("[F3] short_address = full_address")
    else:
        fail("[F3] short_address", f"got: {b['addr_full']}")

    # -------------------------------------------------------
    # Test 4: [F4] district không là chuỗi rỗng
    # -------------------------------------------------------
    svc2 = make_service()
    svc2["addr_district"] = ""   # simulate empty string từ JSON
    am2  = parse_json(svc2["amentities"])
    b2   = build_row_base(svc2, am2)
    if b2["addr_district"] is None:
        ok("[F4] district='' → None")
    else:
        fail("[F4] district empty string", f"got: {b2['addr_district']}")

    # -------------------------------------------------------
    # Test 5: [F5] partner_id không NULL
    # -------------------------------------------------------
    svc3 = make_service()
    am3  = parse_json(svc3["amentities"])
    b3   = build_row_base(svc3, am3)
    if b3["partner_id"] is not None:
        ok("[F5] partner_id không NULL")
    else:
        fail("[F5] partner_id", "vẫn là None")

    # -------------------------------------------------------
    # Test 6: [F6] f_n_b specialty_tags đúng path
    # -------------------------------------------------------
    svc4 = make_service(sector="f_n_b")
    am4  = parse_json(svc4["amentities"])
    aj   = extract_amenities_json(am4, "f_n_b")
    aj_d = json.loads(aj)
    if "specialty_tags" in aj_d.get("facilities", {}):
        ok("[F6] f_n_b có specialty_tags")
    else:
        fail("[F6] f_n_b specialty_tags", f"keys: {list(aj_d.get('facilities', {}).keys())}")

    svc5 = make_service(sector="attraction")
    am5  = parse_json(svc5["amentities"])
    aj5  = json.loads(extract_amenities_json(am5, "attraction"))
    if "specialty_tags" in aj5.get("facilities", {}).get("classification", {}):
        ok("[F6] attraction có classification.specialty_tags")
    else:
        fail("[F6] attraction specialty_tags", f"classification: {aj5.get('facilities', {}).get('classification', {})}")

    # -------------------------------------------------------
    # Test 7: [F7] is_24_7 tính đúng
    # -------------------------------------------------------
    am_24h = make_amentities("f_n_b")
    for day in ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"]:
        am_24h["policies"]["operating_hours"][day] = {
            "status": True,
            "hours": [{"open_time": "00:00", "close_time": "00:00"}]
        }
    oh_24h = extract_operating_hours(am_24h, "f_n_b")
    if oh_24h["is_24_7"] is True:
        ok("[F7] is_24_7 = True khi tất cả ngày 00:00–00:00")
    else:
        fail("[F7] is_24_7", f"expected True, got {oh_24h['is_24_7']}")

    # is_24_7 = False khi 1 ngày đóng cửa
    am_not24 = make_amentities("accommodation")  # sunday = False
    oh_not24 = extract_operating_hours(am_not24, "accommodation")
    if oh_not24["is_24_7"] is False:
        ok("[F7] is_24_7 = False khi có ngày đóng")
    else:
        fail("[F7] is_24_7 False", f"expected False, got {oh_not24['is_24_7']}")

    # -------------------------------------------------------
    # Test 8: [F8] venue_regulations per-sector
    # -------------------------------------------------------
    svc6 = make_service(sector="attraction")
    am6  = parse_json(svc6["amentities"])
    pol6 = extract_policy(am6, "attraction")
    if "gate_and_entry_rules" in (pol6.get("venue_regulations") or {}):
        ok("[F8] attraction.venue_regulations có gate_and_entry_rules")
    else:
        fail("[F8] attraction venue_regulations", f"got: {pol6.get('venue_regulations')}")

    svc7 = make_service(sector="spa_n_wellness")
    am7  = parse_json(svc7["amentities"])
    pol7 = extract_policy(am7, "spa_n_wellness")
    if pol7.get("financial_transparency") is not None:
        ok("[F8] spa.financial_transparency không NULL")
    else:
        fail("[F8] spa financial_transparency", "vẫn là None")

    # -------------------------------------------------------
    # Test 9: sentinel → NULL khi INSERT mới
    # -------------------------------------------------------
    svc8 = make_service(sector="accommodation", story=SENTINEL)
    am8  = parse_json(svc8["amentities"])
    b8   = build_row_base(svc8, am8)
    if b8.get("story") is None:
        ok("[TOAST] sentinel story → None")
    else:
        fail("[TOAST] sentinel", f"story vẫn là: {b8.get('story')}")

    # -------------------------------------------------------
    # Test 10: operating_hours open_time/close_time đúng key
    # -------------------------------------------------------
    am9  = make_amentities("f_n_b")
    oh9  = extract_operating_hours(am9, "f_n_b")
    if oh9["periods"] and oh9["periods"][0].get("open_hour") == 8:
        ok("[F7] open_time=08:00 → open_hour=8")
    else:
        fail("[F7] open_hour", f"periods[0]: {oh9['periods'][0] if oh9['periods'] else 'empty'}")

    # -------------------------------------------------------
    print(f"\n{'='*40}")
    print(f"PASSED: {passed} | FAILED: {failed}")
    if failed == 0:
        print("✅ ALL TESTS PASSED")
    else:
        print(f"❌ {failed} TEST(S) FAILED")
    return failed == 0


# Chạy tests
run_tests()

StatementMeta(, 3af2cce6-0b3e-4e9c-bb6f-2dddbc678c96, 3, Finished, Available, Finished, False)

RUN_ID: bbd6c0da-f721-47e3-afec-dae4c3789714
✓ Cell 2: utility functions ready
✓ Cell 3: sector configs ready
✓ Watermark reset xong — chạy lại toàn bộ notebook từ Cell 1
Watermark cutoff   : 0
Effective cutoff   : 0
Tương đương        : FULL LOAD
Upper bound (seconds): 1787048668
Tương đương          : 2026-08-18 10:24:28
Services to process: 149
✓ Joined rows: 266
✓ Cell 6: transform functions ready
Collected 266 rows
✓ Built 266 poi_master rows
✓ Built 266 ai_poi rows
✓ Built 266 poi_detail rows
⚠ Errors: 0
✓ poi_master DataFrame: 266 rows
✓ MERGE poi_master done
✓ poi_detail DataFrame: 266 rows
✓ MERGE poi_detail done
✓ ai_poi DataFrame: 266 rows
✓ MERGE ai_poi done
✓ Watermark advanced → 1787048668
  Tương đương: 2026-08-18 10:24:28

✅ RUN SUCCESS — RUN_ID: bbd6c0da-f721-47e3-afec-dae4c3789714
   poi_master : 266 rows
   ai_poi     : 266 rows
   poi_detail : 266 rows

=== UNIT TESTS ===

  ✓ make_poi_id deterministic
  ✓ clean() bỏ sentinel → None
  ✓ clean() bỏ chuỗi rỗng → None


True

In [ ]:
# =============================================================================
# CELL 8.3 — SYNC BATCH TO STAGING TABLES (target state AFTER merge)
# =============================================================================
#
# Muc dich: sau khi MERGE xong poi_master / poi_detail, ghi lai dung tap record
# thuoc batch nay (xac dinh qua watermark o Cell 4-5, tuc master_df/detail_df)
# nhung lay GIA TRI THAT da luu trong Gold table (sau khi ap dung COALESCE trong
# MERGE), khong phai gia tri source truoc khi merge.
#
# Cach lam: join lai Gold table bang dung tap poi_id co trong batch (source df),
# roi append toan bo record do (da o trang thai final) vao bang stg_*_sync.
#
# Luu y: broadcast join thay vi collect() + isin() de tranh literal-list qua dai
# khi batch nhieu record.
# =============================================================================

STG_POI_MASTER_SYNC = "lh_vv_gold.dbo.stg_poi_master_sync"
STG_POI_DETAIL_SYNC = "lh_vv_gold.dbo.stg_poi_detail_sync"

if master_rows:
    master_poi_ids_df = master_df.select("poi_id").distinct()

    synced_master_df = (
        spark.table(GLD_POI_MASTER)
        .join(F.broadcast(master_poi_ids_df), "poi_id", "inner")
    )

    (
        synced_master_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(STG_POI_MASTER_SYNC)
    )

    print(f"MERGE poi_master synced -> {STG_POI_MASTER_SYNC}: {master_poi_ids_df.count():,} poi_id")

if detail_rows:
    detail_poi_ids_df = detail_df.select("poi_id").distinct()

    synced_detail_df = (
        spark.table(GLD_POI_DETAIL)
        .join(F.broadcast(detail_poi_ids_df), "poi_id", "inner")
    )

    (
        synced_detail_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(STG_POI_DETAIL_SYNC)
    )

    print(f"MERGE poi_detail synced -> {STG_POI_DETAIL_SYNC}: {detail_poi_ids_df.count():,} poi_id")

In [2]:
%%sql

--  SELECT * FROM lh_vv_gold.dbo.poi_master where ingestion_source ='partner_portal' ;
-- SELECT * FROM lh_vv_gold.dbo.poi_master where ingestion_source ='partner_portal' LIMIT 1000

SELECT * FROM lh_vv_gold.dbo.poi_master where poi_id='3525b1ee-2f90-4f29-911a-c5f26ed3a61f';

-- -- SELECT * FROM lh_vv_bronze.partner.business_services where business_sector= 'accommodation';

-- SELECT * FROM lh_vv_gold.dbo.poi_master WHERE poi_id = '4a100522-9ca1-9479-6c41-823fe6c13820';

-- SELECT
-- *
-- FROM lh_vv_bronze.dbo.partner_raw_data
-- WHERE get_json_object(after,  '$.id') = '08891d14-98f8-4e14-aba1-3ad3e6bca436'
--    OR get_json_object(before, '$.id') = '08891d14-98f8-4e14-aba1-3ad3e6bca436'
-- ORDER BY EventProcessedUtcTime ASC;

-- SELECT
-- *
-- FROM lh_vv_bronze.dbo.partner_raw_data
-- WHERE source like ('%business_service_i18ns%')
-- ORDER BY EventProcessedUtcTime ASC limit 10;

StatementMeta(, e88a3bb9-cf47-4b93-ace6-53afb24d367c, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 28 fields>